# Knowledge Distillation – Kaggle 2×T4 Edition or P100

In [ ]:
# Verify Kaggle GPU environment + capture key versions for the rest of the cells.
import torch, subprocess, sys, os

print(f"Python: {sys.version.split()[0]}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA (torch): {torch.version.cuda}")
print(f"cuDNN: {torch.backends.cudnn.version()}")
print(f"GPU count: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    cc = f"sm_{p.major}{p.minor}"
    print(f"  GPU {i}: {p.name:20s}  {p.total_memory/1e9:5.1f} GB  {cc}")

# nvcc — needed for source-build of mamba-ssm / causal-conv1d.
try:
    out = subprocess.check_output(["nvcc", "--version"], text=True)
    nvcc_line = [l for l in out.splitlines() if "release" in l.lower()]
    print(f"nvcc: {nvcc_line[0].strip() if nvcc_line else out.splitlines()[-1]}")
except Exception as e:
    print(f"nvcc: NOT FOUND ({e}). Source-build of mamba kernels will fail; "
          "fall back to slow Python path or use prebuilt wheels.")

# Fail fast if T4 (sm_75) — mamba-ssm requires sm_70+ but P100 (sm_60) is unsupported.
if torch.cuda.device_count() > 0:
    cc = torch.cuda.get_device_capability(0)
    assert cc[0] >= 7, (
        f"GPU compute capability {cc[0]}.{cc[1]} is too old for mamba-ssm CUDA kernels. "
        "Switch the Kaggle accelerator to 'GPU T4 x2' (sm_75) instead of P100 (sm_60)."
    )


## Install requirements

In [ ]:
!apt-get install -y -qq ffmpeg libavcodec-extra > /dev/null 2>&1
# Don't pin transformers — the Kaggle default (currently v5.x) works with
# the mamba-ssm fast-path kernels. A hard pin gets silently overridden by
# huggingface_hub / openai-whisper anyway, so it just adds confusion.
!pip install -q -U transformers
!pip install -q "datasets[audio]<4.0.0"
!pip install -q sentencepiece jiwer evaluate
!pip install -q WeTextProcessing
!pip install -q -U openai-whisper
!pip install -q --upgrade huggingface_hub


In [ ]:
# ── Mamba CUDA kernels — auto-resolve for the actual env ─────────
import sys, torch, importlib, subprocess

cc_major = torch.cuda.get_device_capability(0)[0]
assert cc_major >= 7, (
    f"GPU compute capability {cc_major}.x is too old for mamba-ssm CUDA kernels. "
    "Switch the Kaggle accelerator to 'GPU T4 x2' (sm_75) instead of P100 (sm_60)."
)

# --no-build-isolation
!pip install -q --no-build-isolation ninja packaging wheel
!pip install -q --no-build-isolation causal-conv1d
!pip install -q --no-build-isolation mamba-ssm


def _check():
    failures = []
    try:
        from causal_conv1d import causal_conv1d_fn, causal_conv1d_update  # noqa
    except Exception as e:
        failures.append(f"causal_conv1d import failed: {e}")
    try:
        from mamba_ssm.ops.selective_scan_interface import (
            selective_scan_fn, mamba_inner_fn,
        )  # noqa
        from mamba_ssm.ops.triton.selective_state_update import selective_state_update  # noqa
    except Exception as e:
        failures.append(f"mamba_ssm import failed: {e}")
    return failures

problems = _check()
if problems:
    print("Fast-path NOT available. Reasons:")
    for p in problems:
        print(" -", p)
    raise RuntimeError("Mamba CUDA kernels did not import; fix before continuing.")
print("mamba-ssm + causal-conv1d kernels importable — fast path enabled")

In [ ]:
# Should not raise ImportError
from mamba_ssm.ops.selective_scan_interface import mamba_inner_fn, selective_scan_fn
from mamba_ssm.ops.triton.selective_state_update import selective_state_update

In [ ]:
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("huggingface_token")
wandb_key = user_secrets.get_secret("wandb_api_key")
login(token=hf_token)

## Config pipeline

In [ ]:
import os, gc, json, math, time, random
import numpy as np
from pathlib import Path
from dataclasses import dataclass, field
from typing import List
import torch, torchaudio
import torch
import functools
from pathlib import Path
from transformers import WhisperForConditionalGeneration, WhisperProcessor, AutoModelForSpeechSeq2Seq, AutoProcessor, MambaForCausalLM, MambaConfig
from datasets import load_dataset, Audio
import datasets
from tqdm import tqdm

from torch.utils.data import Dataset, DataLoader
import sentencepiece as spm

print = functools.partial(print, flush=True)
print(datasets.__version__)

# Hardware-aware helpers
if torch.cuda.device_count() > 0:
    NUM_GPUS = torch.cuda.device_count()
    GPU_MEM_GB = torch.cuda.get_device_properties(0).total_memory / 1e9
    IS_T4 = "T4" in torch.cuda.get_device_name(0)


@dataclass
class Config:
    # Storage (Kaggle working dir, persists across saves)
    root: str = "/kaggle/working/edge_asr"

    # Teacher
    teacher_id: str = "openai/whisper-large-v3"
    # T4 does not support flash_attention_2
    teacher_attn: str = "sdpa"
    teacher_d: int = 1280

    # Pseudo Labeling
    psuedo_batch: int = 8
    max_label_len: int = 128
    max_samples_per_dataset: int = 30_000

    # Filtering config
    # loop filter
    max_repeats: int = 3
 
    # glitch filter
    max_word_len: int = 20
 
    # speed filter (words per second)
    min_wps: float = 0.5
    max_wps: float = 6.0
 
    # WER/CER gate (percentage)
    wer_threshold: float = 10.0
 
    # all-caps minimum length
    allcaps_min_len: int = 5

    # WER threshold
    wer_threshold: float = 10.0

    # Tokenizer
    spm_vocab: int = 5000
    spm_coverage: float = 0.9995
    vocab_size: int = 5001

    # Student
    mamba_pretrained: str = "state-spaces/mamba-130m-hf"
    mamba_d: int = 768
    cnn_ch: int = 768
    cnn_ks: int = 5
    n_mels: int = 80
    sr: int = 16000

    # Loss
    a_kl: float = 1.0
    a_ctc: float = 0.3
    # temp: float = 2.0

    # Frozen stage — pure feature distillation (a_ctc=0 here; see train_stage).
    # 1 epoch is enough: loss_kl saturates near 0.004 within the first epoch,
    # and additional epochs refine kl_head while ctc_head sits at random init.
    # The remaining session budget is more valuable spent in the unfrozen stage
    # where CTC alignment actually gets learned.
    fr_epochs: int = 1
    fr_lr: float = 3e-4
    fr_bs: int = 16 # batch size
    fr_ga: int = 2
    max_s1: float = 10.0

    # Unfrozen stage
    un_epochs: int = 6
    un_lr: float = 5e-5
    un_bs: int = 4
    un_ga: int = 8
    gc_norm: float = 1.0
    warmup: int = 500
    max_s2: float = 30.0

    # Datasets
    datasets: List[tuple] = field(default_factory=lambda: [
        # ('facebook/multilingual_librispeech', 'french',  'train', 'transcript', 'french',  True),
        # ('facebook/multilingual_librispeech', 'spanish', 'train', 'transcript', 'spanish', True),
        # ('facebook/multilingual_librispeech', 'german', 'train', 'transcript', 'german',  True),
        # ('fsicoli/common_voice_22_0', 'vi', 'train', 'sentence', 'vietnamese', True),
        # ('fsicoli/common_voice_22_0', 'ja', 'train', 'sentence', 'japanese',   True),
        # ('fsicoli/common_voice_22_0', 'ko', 'train', 'sentence', 'korean',     True),
        # ('fsicoli/common_voice_22_0', 'zh-CN', 'train', 'sentence', 'chinese', True),
        ('fsicoli/common_voice_22_0', 'en', 'train', 'sentence', 'english', True),
        # ('nguyendv02/ViMD_Dataset', 'default', 'train', 'text', 'vietnamese', True),
        # ('pnnbao-ump/VieNeu-TTS', 'default', 'train', 'text', 'vietnamese', True)
    ])

    # Regularization
    dropout: float = 0.1   # applied between Mamba output and heads

    # Early stopping (per-stage, monitors val/loss after each epoch)
    es_enabled: bool = True
    es_patience: int = 3    # epochs without improvement before stopping
    es_min_delta: float = 1e-4  # minimum drop in val metric to count as improvement
    es_metric: str = 'loss'     # 'loss' (val total loss) or 'wer'

    seed: int = 42
    workers: int = 2      # Kaggle has fewer CPU cores than Colab HM
    log_every: int = 50

    def __post_init__(self):
        # Read-only Kaggle input mounts (provided datasets).
        self.pseudo_dir  = '/kaggle/input/datasets/leviettrieu369/pseudo-labeling'
        self.filter_dir  = '/kaggle/input/datasets/leviettrieu369/filtering-transcription'
        self.cache_dir   = '/kaggle/input/datasets/leviettrieu369/teacher-cache/teacher_cache'
        self.spm_prefix  = '/kaggle/input/datasets/leviettrieu369/tokenizer/spm'

        # Checkpoints: source is read-only (the dataset that holds baseline checkpoints from prior sessions)
        # self.ckpt_load_dir = '/kaggle/input/datasets/leviettrieu369/distillation-checkpoint/edge_asr/checkpoints'
        self.ckpt_load_dir = f'{self.root}/checkpoints'
        self.ckpt_dir      = f'{self.root}/checkpoints'

        # Loss log
        self.loss_log_path = f'{self.root}/training_log.jsonl'

        self.onnx_path = f'{self.root}/student.onnx'

    def safe_name(self, i):
        n, c = self.datasets[i][0].split('/')[-1], self.datasets[i][1]
        return f'{n}_{c}'


C = Config()
# Only mkdir writable destinations. /kaggle/input is a read-only FUSE mount;
# calling makedirs on it raises OSError even with exist_ok=True if the path
# doesn't already exist on the host.
def _is_writable_path(p):
    return not p.startswith('/kaggle/input')

for d in [C.root, C.ckpt_dir, os.path.dirname(C.loss_log_path)]:
    if _is_writable_path(d):
        os.makedirs(d, exist_ok=True)

# Sanity report on checkpoint source availability.
if os.path.isdir(C.ckpt_load_dir):
    _existing = [f for f in os.listdir(C.ckpt_load_dir) if f.endswith('.pt')]
    print(f'Checkpoint source (read-only): {C.ckpt_load_dir}')
    print(f'  Found {len(_existing)} .pt file(s): {_existing[:5]}{"..." if len(_existing) > 5 else ""}')
else:
    print(f'Checkpoint source not found (will train from scratch): {C.ckpt_load_dir}')
print(f'Checkpoint destination (writable): {C.ckpt_dir}')
print(f'Loss log path:                     {C.loss_log_path}')

random.seed(C.seed)


def flush():
    """Free GPU memory on ALL devices."""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()
        for i in range(torch.cuda.device_count()):
            alloc = torch.cuda.memory_allocated(i) / 1e9
            total = torch.cuda.get_device_properties(i).total_memory / 1e9
            print(f'  [GPU {i}] {alloc:.1f}/{total:.0f} GB')


print(f'{len(C.datasets)} datasets, root: {C.root}')
if torch.cuda.device_count() > 0:
    print(f'Hardware : {NUM_GPUS}× {torch.cuda.get_device_name(0)}')
print(f'Precision: float16  |  Attention: {C.teacher_attn}')
print(f'Frozen: bs={C.fr_bs} × GA={C.fr_ga} = {C.fr_bs*C.fr_ga} eff')
print(f'Unfrozen: bs={C.un_bs} × GA={C.un_ga} = {C.un_bs*C.un_ga} eff')

In [ ]:
# Pre-flight: verify the read-only Kaggle input dataset is mounted.
# These paths come from Config (cell above) and MUST exist before training,
# otherwise KDDataset.__init__ will crash deep into the training cell after
# 5+ minutes of setup.
import os

_required = {
    "teacher cache (shards)" : C.cache_dir,
    "SPM tokenizer model"    : C.spm_prefix + ".model",
}
_optional = {
    "pseudo labels"          : C.pseudo_dir,
    "filtered transcripts"   : C.filter_dir,
    "baseline checkpoints"   : C.ckpt_load_dir,
}

print("─ Required inputs ─")
_missing = []
for name, path in _required.items():
    ok = os.path.exists(path)
    print(f"  [{'OK' if ok else 'MISSING'}] {name:25s} {path}")
    if not ok:
        _missing.append((name, path))

print("\n─ Optional inputs (training works without them) ─")
for name, path in _optional.items():
    ok = os.path.exists(path)
    print(f"  [{'OK' if ok else '----'}] {name:25s} {path}")

if _missing:
    msg = (
        "\nMissing required Kaggle input(s):\n"
        + "\n".join(f"  - {n}: {p}" for n, p in _missing)
        + "\n\nAttach the dataset(s) via the Kaggle UI: "
        "Notebook  : Add Input  : search 'leviettrieu369' (or your own copy)."
    )
    raise FileNotFoundError(msg)

# Quick count of cache shards so we know data is real, not just an empty dir.
if os.path.isdir(C.cache_dir):
    _lang_dirs = [d for d in os.listdir(C.cache_dir)
                  if os.path.isdir(os.path.join(C.cache_dir, d))]
    _total_shards = 0
    for d in _lang_dirs:
        _total_shards += sum(1 for f in os.listdir(os.path.join(C.cache_dir, d))
                             if f.endswith(".npz"))
    print(f"\nTeacher cache: {len(_lang_dirs)} language dir(s), {_total_shards} shards total")
    if _total_shards == 0:
        raise RuntimeError(f"No .npz shards found under {C.cache_dir} — cache is empty.")


## Pseudo Labelling

The teacher (Whisper-large-v3, ~3 GB fp16) is loaded with
`device_map="auto"` so Accelerate shards it across both T4s.  
Batch size is reduced from 32  : 8 to stay within 16 GB per GPU.

In [ ]:
# data type for T4
dtype = torch.float16
BATCH_SIZE = C.psuedo_batch
MAX_SAMPLES_PER_DS = C.max_samples_per_dataset

print(f'Loading {C.teacher_id} with device_map="auto" ...')
t_model = AutoModelForSpeechSeq2Seq.from_pretrained(
    C.teacher_id,
    dtype=dtype,
    low_cpu_mem_usage=True,
    use_safetensors=True,
    attn_implementation=C.teacher_attn,
    device_map="auto",
)
proc = AutoProcessor.from_pretrained(C.teacher_id)

# Silence warnings
t_model.generation_config.max_length = None
t_model.generation_config.suppress_tokens = None
t_model.generation_config.begin_suppress_tokens = None

# # Verify sharding
if hasattr(t_model, 'hf_device_map'):
    devices_used = set(str(v) for v in t_model.hf_device_map.values())
    print(f'Teacher sharded across devices: {devices_used}')
flush()


@torch.no_grad()
def process_batch(batch_samples, batch_meta, fout, gen_kw):
    """Run batched inference using model.generate() directly (no pipeline)."""
    # Extract raw audio arrays
    audio_arrays = [s['raw'] for s in batch_samples]
    
    # Process through feature extractor
    inputs = proc.feature_extractor(
        audio_arrays,
        sampling_rate=C.sr,
        return_tensors="pt",
        padding=True,
    )
    
    # Move to the model's first device & cast dtype
    first_device = next(iter(t_model.hf_device_map.values()))
    input_features = inputs.input_features.to(
        device=f"cuda:{first_device}" if isinstance(first_device, int) else first_device,
        dtype=dtype,
    )
    
    # Generate
    predicted_ids = t_model.generate(
        input_features,
        **gen_kw,
    )
    
    # Decode
    texts = proc.tokenizer.batch_decode(predicted_ids, skip_special_tokens=True)
    
    written = 0
    for text, meta in zip(texts, batch_meta):
        wt = text.strip()
        if not wt:
            continue
        meta['whisper'] = wt
        fout.write(json.dumps(meta, ensure_ascii=False) + '\n')
        written += 1
    return written


for di, (ds_name, ds_cfg, ds_split, text_col, lang, _) in enumerate(C.datasets):
    safe = C.safe_name(di)
    jsonl = f'{C.pseudo_dir}/{safe}.jsonl'
    done_flag = f'{C.pseudo_dir}/{safe}.done'

    if os.path.exists(done_flag):
        print(f'[{di+1}] {safe} — already done')
        continue

    print(f'\n[{di+1}/{len(C.datasets)}] {safe} ({lang})')
    ds = load_dataset(ds_name, ds_cfg, split=ds_split, streaming=True, trust_remote_code=True)
    ds = ds.take(MAX_SAMPLES_PER_DS)
    # ds = ds.cast_column('audio', Audio(sampling_rate=C.sr))
    ds = ds.select_columns(['audio', text_col])
    ds = ds.cast_column('audio', Audio(sampling_rate=C.sr))

    # Resume support
    done_count = 0
    if os.path.exists(jsonl):
        with open(jsonl) as f:
            done_count = sum(1 for _ in f)
        print(f'  Resuming from {done_count}')
        # if done_count >= MAX_SAMPLES_PER_DS:
        #     print(f'  Already have {done_count} ≥ {MAX_SAMPLES_PER_DS}, skipping.')
        #     Path(done_flag).touch()
        #     continue
    if done_count > 0:
        ds = ds.skip(done_count)

    fout = open(jsonl, 'a', buffering=1)
    gen_kw = {
        'max_new_tokens': C.max_label_len,
        'language': lang,
        'task': 'transcribe',
    }
    errors = 0
    total_written = done_count
    skipped = 0

    batch_samples = []
    batch_meta = []

    for i, sample in enumerate(ds):
        idx = i + done_count
        try:
            audio = sample['audio']
            duration = round(len(audio['array']) / audio['sampling_rate'], 3)

            if duration > 30:
                skipped += 1
                continue

            batch_samples.append({
                'raw': audio['array'],
                'sampling_rate': audio['sampling_rate'],
            })
            batch_meta.append({
                'idx': idx,
                'original': sample.get(text_col, ''),
                'duration': duration,
                'lang': lang,
            })

            if len(batch_samples) >= BATCH_SIZE:
                total_written += process_batch(
                    batch_samples, batch_meta, fout, gen_kw
                )
                batch_samples.clear()
                batch_meta.clear()
                if (idx + 1) % 200 == 0:
                    print(
                        f'    {idx+1} processed | '
                        f'{total_written} saved | '
                        f'{skipped} skipped | '
                        f'{errors} errors'
                    )

        except Exception as e:
            errors += 1
            print(f'  [ERR] sample {idx}: {e}')
            batch_samples.clear()
            batch_meta.clear()
            if errors > 20:
                print('  Too many errors, stopping this dataset.')
                break
            continue

    # Flush remaining
    if batch_samples:
        try:
            total_written += process_batch(
                batch_samples, batch_meta, fout, gen_kw
            )
        except Exception as e:
            print(f'  [ERR] final batch: {e}')

    fout.close()
    Path(done_flag).touch()
    count = sum(1 for _ in open(jsonl))
    print(f'  {safe}: {count} pseudo-labels saved, {errors} errors')

del t_model, proc
flush()

## Heuristic filtering

In [ ]:
# !pip install -q WeTextProcessing # for chinese
# !pip install -U -q openai-whisper

In [ ]:
import json
import re
import unicodedata

from evaluate import load as load_metric
from whisper.normalizers.basic import BasicTextNormalizer

# ─── Regex patterns ──────────────────────────────────────────────────
# CJK & fullwidth punctuation, general punctuation, ASCII punctuation
_CJK_PUNCT = re.compile(
    r'[\u3000-\u303F\uFF00-\uFFEF\u2000-\u206F\u2E00-\u2E7F'
    r'!"#$%&\'()*+,\-./:;<=>?@\[\]^_`{|}~]'
)
_MULTI_SPACE = re.compile(r'\s+')

# Character-class detectors
_CJK_IDEOGRAPH = re.compile(r'([\u4E00-\u9FFF\u3400-\u4DBF\uF900-\uFAFF])')
_HIRAGANA      = re.compile(r'([\u3040-\u309F])')
_KATAKANA      = re.compile(r'([\u30A0-\u30FF\u31F0-\u31FF])')
_HANGUL        = re.compile(r'([\uAC00-\uD7AF\u1100-\u11FF\u3130-\u318F])')

# Quick test: does the text contain any CJK / Kana / Hangul?
_HAS_CJK = re.compile(
    r'[\u4E00-\u9FFF\u3400-\u4DBF\uF900-\uFAFF'   # CJK ideographs
    r'\u3040-\u309F'                                 # Hiragana
    r'\u30A0-\u30FF\u31F0-\u31FF'                    # Katakana
    r'\uAC00-\uD7AF\u1100-\u11FF\u3130-\u318F]'     # Hangul
)

# Language tag  : language code mapping
_LANG_TAGS = {
    '_zh-CN': 'zh', '_zh-TW': 'zh', '_zh': 'zh',
    '_ja': 'ja',
    '_ko': 'ko',
}

# Languages that use CJK scripts (no spaces between words)
_CJK_LANGS = {'zh', 'ja', 'ko'}

# Languages that use Latin script (case-sensitive)
_LATIN_LANGS = {'en', 'fr', 'es', 'de', 'vi'}


# ─── Normalization ───────────────────────────────────────────────────
def _basic_clean(text: str) -> str:
    """NFKC normalize, lowercase, strip CJK/ASCII punctuation."""
    text = unicodedata.normalize('NFKC', text)
    text = text.lower()
    text = _CJK_PUNCT.sub('', text)
    return _MULTI_SPACE.sub(' ', text).strip()


def _char_tokenize(text: str, pattern: re.Pattern) -> str:
    """Insert spaces around each character matched by *pattern*."""
    return _MULTI_SPACE.sub(' ', pattern.sub(r' \1 ', text)).strip()


def normalize_chinese(text: str) -> str:
    """Normalize Chinese: clean  : space-separate each CJK ideograph."""
    text = _basic_clean(text)
    return _char_tokenize(text, _CJK_IDEOGRAPH)


def normalize_japanese(text: str) -> str:
    """Normalize Japanese: clean  : space-separate CJK, Hiragana, Katakana."""
    text = _basic_clean(text)
    text = text.replace('\u2015', '\u30FC')  # horizontal bar  : prolonged sound mark
    for pat in (_CJK_IDEOGRAPH, _HIRAGANA, _KATAKANA):
        text = pat.sub(r' \1 ', text)
    return _MULTI_SPACE.sub(' ', text).strip()


def normalize_korean(text: str) -> str:
    """Normalize Korean: clean  : space-separate Hangul syllables."""
    text = _basic_clean(text)
    return _char_tokenize(text, _HANGUL)


_CJK_NORMALIZERS = {
    'zh': normalize_chinese,
    'ja': normalize_japanese,
    'ko': normalize_korean,
}


def detect_lang_from_name(safe_name: str) -> str | None:
    """Detect CJK language code from the dataset safe_name."""
    for tag, lang in _LANG_TAGS.items():
        if tag in safe_name:
            return lang
    return None


# ─── Heuristic filters ───────────────────────────────────────────────
def _is_cjk_text(text: str) -> bool:
    """Return True if the text contains any CJK / Kana / Hangul characters."""
    return bool(_HAS_CJK.search(text))


def max_consecutive_repeats(text: str, is_cjk: bool = False) -> int:
    """H1: longest run of identical consecutive tokens.

    For CJK text, tokenize at the character level (each char is a token).
    For Latin text, tokenize at the word level (split on whitespace).
    """
    if is_cjk:
        tokens = list(text.lower().replace(' ', ''))
    else:
        tokens = text.lower().split()
    if len(tokens) <= 1:
        return 1
    max_run = cur_run = 1
    for i in range(1, len(tokens)):
        if tokens[i] == tokens[i - 1]:
            cur_run += 1
            if cur_run > max_run:
                max_run = cur_run
        else:
            cur_run = 1
    return max_run


def longest_word_length(text: str, is_cjk: bool = False) -> int:
    """H2: character length of the longest token.

    Skipped for CJK languages because they don't use space-delimited words;
    the concept of "word length" is meaningless.
    """
    if is_cjk:
        return 0  # always pass — not applicable
    words = text.split()
    return max((len(w) for w in words), default=0)


def chars_per_second(text: str, duration: float) -> float:
    """H3 (CJK): speech rate measured in characters per second.

    Counts only non-space characters.
    """
    if duration <= 0:
        return float('inf')
    n_chars = len(text.replace(' ', ''))
    return n_chars / duration


def words_per_second(text: str, duration: float) -> float:
    """H3 (Latin): speech rate measured in words per second."""
    if duration <= 0:
        return float('inf')
    return len(text.split()) / duration


def is_allcaps_garbage(text: str, min_len: int = 5) -> bool:
    """H5: True if the entire text is uppercase and longer than min_len.

    Only meaningful for Latin scripts — CJK characters have no case
    distinction, so `.upper() == text` is always True. This function
    now returns False for any text containing CJK characters.
    """
    if _is_cjk_text(text):
        return False
    return len(text) > min_len and text.upper() == text


# ─── Filter statistics ───────────────────────────────────────────────
@dataclass
class FilterStats:
    """Per-dataset rejection counters for diagnostics."""
    total: int = 0
    kept: int = 0
    rejected_empty: int = 0
    rejected_h1: int = 0
    rejected_h2: int = 0
    rejected_h3: int = 0
    rejected_h4: int = 0
    rejected_h5: int = 0

    @property
    def dropped(self) -> int:
        return self.total - self.kept

    @property
    def drop_pct(self) -> float:
        return self.dropped / max(self.total, 1) * 100

    def summary(self) -> str:
        lines = [
            f"  total:    {self.total}",
            f"  kept:     {self.kept} ({100 - self.drop_pct:.1f}%)",
            f"  dropped:  {self.dropped} ({self.drop_pct:.1f}%)",
            f"    empty/missing:   {self.rejected_empty}",
            f"    H1 (repeats):    {self.rejected_h1}",
            f"    H2 (word len):   {self.rejected_h2}",
            f"    H3 (speed):      {self.rejected_h3}",
            f"    H4 (WER/CER):    {self.rejected_h4}",
            f"    H5 (all-caps):   {self.rejected_h5}",
        ]
        return '\n'.join(lines)

In [ ]:
def filter_dataset(
    src: str | Path,
    dst: str | Path,
    safe_name: str,
    cfg: Config,
    wer_metric,
    cer_metric,
    en_normalizer,
    show_progress: bool = True,
) -> FilterStats:
    """Filter a pseudo-labelled JSONL file using language-aware heuristics.

    For CJK languages (zh, ja, ko):
      - H1 operates on individual characters instead of words
      - H2 (word length) is skipped entirely
      - H3 uses characters-per-second instead of words-per-second
      - H4 uses CER (Character Error Rate) instead of WER
      - H5 (all-caps) is skipped (CJK has no case)

    For Latin-script languages the original behaviour is preserved.
    """
    lang = detect_lang_from_name(safe_name)
    is_cjk = lang in _CJK_LANGS
    normalize = _CJK_NORMALIZERS.get(lang, en_normalizer)
    # Use CER for CJK, WER for everything else
    error_metric = cer_metric if is_cjk else wer_metric
    stats = FilterStats()

    # CJK-specific speed bounds (characters per second)
    cjk_min_cps = 1.0
    cjk_max_cps = 12.0

    # Count lines for progress bar
    n_lines = 0
    if show_progress:
        with open(src) as f:
            n_lines = sum(1 for _ in f)

    try:
        from tqdm.auto import tqdm
        def _iter(it):
            return tqdm(it, total=n_lines, desc=safe_name,
                        leave=False) if show_progress else it
    except ImportError:
        def _iter(it):
            return it

    with open(src) as fin, open(dst, 'w') as fout:
        for line in _iter(fin):
            stats.total += 1
            entry = json.loads(line)

            gt = entry.get('original', '')
            wt = entry.get('whisper', '')
            duration = entry.get('duration', 0.0)

            # Empty check
            if not gt or not wt:
                stats.rejected_empty += 1
                continue

            # H5: all-caps garbage (Latin-only)
            if is_allcaps_garbage(wt, cfg.allcaps_min_len):
                stats.rejected_h5 += 1
                continue

            # H1: repeated tokens
            if max_consecutive_repeats(wt, is_cjk=is_cjk) >= cfg.max_repeats:
                stats.rejected_h1 += 1
                continue

            # H2: abnormal word length (Latin-only)
            if longest_word_length(wt, is_cjk=is_cjk) > cfg.max_word_len:
                stats.rejected_h2 += 1
                continue

            # H3: speech-rate sanity
            if duration > 0:
                if is_cjk:
                    cps = chars_per_second(wt, duration)
                    if cps < cjk_min_cps or cps > cjk_max_cps:
                        stats.rejected_h3 += 1
                        continue
                else:
                    wps = words_per_second(wt, duration)
                    if wps < cfg.min_wps or wps > cfg.max_wps:
                        stats.rejected_h3 += 1
                        continue

            # Normalize for error-rate computation
            ref = normalize(gt)
            hyp = normalize(wt)

            if not ref or not hyp:
                stats.rejected_empty += 1
                continue

            # H4: WER / CER gate
            score = 100.0 * error_metric.compute(
                predictions=[hyp], references=[ref]
            )
            if score >= cfg.wer_threshold:
                stats.rejected_h4 += 1
                continue

            fout.write(line)
            stats.kept += 1

    return stats

In [ ]:
en_norm = BasicTextNormalizer()
wer_m = load_metric('wer')
cer_m = load_metric('cer')
all_stats = {}

for di in range(len(C.datasets)):
    safe = C.safe_name(di)
    src = f'{C.pseudo_dir}/{safe}.jsonl'
    dst = f'{C.filter_dir}/{safe}.jsonl'

    # if os.path.exists(dst):
    #     print(f"skip {dst}")
    #     continue

    stats = filter_dataset(
        src=src,
        dst=dst,
        safe_name=safe,
        cfg=C,
        wer_metric=wer_m,
        cer_metric=cer_m,
        en_normalizer=en_norm,
    )

    all_stats[safe] = stats

    print(f'\n[{di + 1}] {safe}:')
    print(stats.summary())

agg = FilterStats()
for s in all_stats.values():
    for f in ('total', 'kept', 'rejected_empty',
              'rejected_h1', 'rejected_h2', 'rejected_h3',
              'rejected_h4', 'rejected_h5'):
        setattr(agg, f, getattr(agg, f) + getattr(s, f))

print('=== Aggregate across all datasets ===')
print(agg.summary())

## Train SentencePiece
Unigram tokenizer on all filtered text. CPU only.

In [ ]:
import sentencepiece as spm

model_file = C.spm_prefix + '.model'

if os.path.exists(model_file):
    sp = spm.SentencePieceProcessor()
    sp.Load(model_file)
    print(f'Exists: {sp.GetPieceSize()} tokens')
else:
    txt = os.path.join(os.path.dirname(C.spm_prefix), 'all.txt')
    with open(txt, 'w', encoding='utf-8') as f:
        for di in range(len(C.datasets)):
            p = f'{C.filter_dir}/{C.safe_name(di)}.jsonl'
            # if not os.path.exists(p):
            #     continue
            n=0
            for line in open(p):
                t = json.loads(line).get('whisper','')
                if t:
                    f.write(t.strip()+'\n')
                    n += 1
                if n >= 500_000:
                    break
            print(f'{C.safe_name(di)}:{n}')

    spm.SentencePieceTrainer.Train(
        input=txt, 
        model_prefix=C.spm_prefix,
        vocab_size=C.spm_vocab,
        model_type='unigram',
        character_coverage=C.spm_coverage,
        split_by_whitespace=False,
        unk_id=0,
        bos_id=-1,
        eos_id=-1,
        pad_id=3,
        normalization_rule_name='nmt_nfkc_cf',
        num_threads=os.cpu_count(),
        input_sentence_size=5_000_000,
        shuffle_input_sentence=True)
    sp = spm.SentencePieceProcessor()
    sp.Load(model_file)

C.vocab_size = sp.GetPieceSize() + 1
print(f'vocab = {C.vocab_size} (incl CTC blank)')
for lang, t in [('fr','bonjour le monde'),('vi','xin chào thế giới'),
                ('ja','音声認識システム'),('ko','음성 인식')]:
    print(f'  [{lang}] {sp.EncodeAsPieces(t)} {len(sp.EncodeAsPieces(t))} tokens')

## Cache Teacher Encoder
Runs encoder on filtered audio, saves `.npz` shards.  
Teacher only (~6 GB).

In [ ]:
from transformers import WhisperForConditionalGeneration, WhisperFeatureExtractor
from tqdm.auto import tqdm

device = torch.device('cuda')
print(f'Loading encoder...')
t_model = WhisperForConditionalGeneration.from_pretrained(
    C.teacher_id, torch_dtype=torch.bfloat16, low_cpu_mem_usage=True,
    use_safetensors=True, attn_implementation=C.teacher_attn).to(device)
t_model.eval()
enc = t_model.get_encoder()

# Use Whisper's own feature extractor (128 mel bins for large-v3)
feat_ext = WhisperFeatureExtractor.from_pretrained(C.teacher_id)
WHISPER_N_MELS = feat_ext.feature_size   # 128 for large-v3
print(f'Encoder loaded | mel bins: {WHISPER_N_MELS}')


# Helpers
def quantize_to_int8(arr_f16):
    """Per-sample min/max int8 quantization. Returns (int8_arr, scale, zero)."""
    fmin, fmax = arr_f16.min(), arr_f16.max()
    if fmax - fmin < 1e-8:
        return np.zeros_like(arr_f16, dtype=np.int8), np.float32(1.0), np.float32(0.0)
    
    scale = np.float32((fmax - fmin) / 254.0)
    # CORRECTED: zero point maps fmin to -127
    zero = np.float32(np.round(-127.0 - (fmin / scale)))
    
    # CORRECTED: add the zero point, don't subtract it
    q = np.clip(np.round(arr_f16 / scale + zero), -127, 127).astype(np.int8)
    return q, scale, zero


def save_shard(hidden_chunks, scales_list, zeros_list, mel_lens_list,
               texts_list, offsets, out_path):
    """Save one ragged int8 shard to disk."""
    np.savez_compressed(out_path,
        hidden_cat=np.concatenate(hidden_chunks, axis=0),
        offsets=np.array(offsets, dtype=np.int32),
        scales=np.array(scales_list, dtype=np.float32),
        zeros=np.array(zeros_list, dtype=np.float32),
        mel_lens=np.array(mel_lens_list, dtype=np.int32),
        texts=np.array(texts_list))
    return os.path.getsize(out_path)


def encode_sample(raw_audio):
    """Feature-extract + encoder forward  : (h_trimmed_np, mel_len)."""
    inputs = feat_ext(
        raw_audio, sampling_rate=C.sr,
        return_tensors="pt", padding="max_length",
        max_length=480000,
    )
    mel = inputs.input_features
    n_samples = len(raw_audio)
    mel_len = min(n_samples // 160, 3000)

    with torch.no_grad():
        h = enc(mel.to(device, dtype=torch.bfloat16)).last_hidden_state
        h_np = h.cpu().float().numpy()[0]

    enc_len = max(1, mel_len // 2)
    return h_np[:enc_len], mel_len


def flush_shard(buf, lang_dir, shard_idx):
    """Write buffered samples as a shard. Returns bytes written."""
    hidden_chunks, scales_list, zeros_list = [], [], []
    mel_lens_list, texts_list = [], []
    offsets = [0]

    for h_trimmed, mel_len, text in buf:
        q, scale, zero = quantize_to_int8(h_trimmed)
        hidden_chunks.append(q)
        scales_list.append(scale)
        zeros_list.append(zero)
        mel_lens_list.append(mel_len)
        texts_list.append(text)
        offsets.append(offsets[-1] + h_trimmed.shape[0])

    out_path = os.path.join(lang_dir, f'shard_{shard_idx:04d}.npz')
    return save_shard(hidden_chunks, scales_list, zeros_list,
                      mel_lens_list, texts_list, offsets, out_path)


# Per-language streaming cache
SS = 1000   # samples per shard
total_bytes = 0

for di, (ds_name, ds_cfg, ds_split, text_col, lang, _) in enumerate(C.datasets):
    safe = C.safe_name(di)
    fp = f'{C.filter_dir}/{safe}.jsonl'
    if not os.path.exists(fp):
        print(f'  [{di+1}] {safe}: no filtered file, skipping')
        continue

    lang_dir = os.path.join(C.cache_dir, safe)
    os.makedirs(lang_dir, exist_ok=True)
    done_flag = os.path.join(lang_dir, '.done')

    if os.path.exists(done_flag):
        n_existing = len([f for f in os.listdir(lang_dir) if f.endswith('.npz')])
        print(f'  [{di+1}] {safe}: already done ({n_existing} shards)')
        continue

    # Load filtered entries  : build needed-index set
    entries = []
    for line in open(fp):
        entries.append(json.loads(line))
    needed_indices = {e['idx'] for e in entries}
    idx_to_pos = {e['idx']: pos for pos, e in enumerate(entries)}
    max_idx = max(needed_indices) if needed_indices else 0
    print(f'\n[{di+1}/{len(C.datasets)}] {safe} ({lang}): {len(entries)} filtered, '
          f'max_idx={max_idx}')

    # Resume: count completed samples from existing shards
    existing_shards = sorted(f for f in os.listdir(lang_dir) if f.endswith('.npz'))
    n_done_samples = 0
    for sf in existing_shards:
        data = np.load(os.path.join(lang_dir, sf), allow_pickle=True)
        n_done_samples += len(data['scales'])
    shard_idx = len(existing_shards)

    if n_done_samples >= len(entries):
        print(f'  All samples cached ({shard_idx} shards)')
        Path(done_flag).touch()
        continue
    if n_done_samples > 0:
        print(f'  Resuming: {n_done_samples}/{len(entries)} samples done ({shard_idx} shards)')

    # Positions already cached
    cached_positions = set(range(n_done_samples))

    # Stream dataset + process only needed samples
    # select_columns before cast to avoid CastError from extra columns
    # (e.g. sentence_id, sentence_domain in common_voice)
    ds_stream = load_dataset(ds_name, ds_cfg, split=ds_split,
                             streaming=True, trust_remote_code=True)
    ds_stream = ds_stream.select_columns(['audio'])
    ds_stream = ds_stream.cast_column('audio', Audio(sampling_rate=C.sr))

    buf = []  # buffer: list of (h_trimmed, mel_len, text)
    n_processed = n_done_samples
    lang_bytes = 0
    errors = 0

    pbar = tqdm(enumerate(ds_stream), desc=f'{safe}',
                total=max_idx + 1, unit='sample')
    for stream_idx, sample in pbar:
        if stream_idx > max_idx:
            break
        if stream_idx not in needed_indices:
            continue
        pos = idx_to_pos[stream_idx]
        if pos in cached_positions:
            continue

        try:
            raw_audio = sample['audio']['array']
            h_trimmed, mel_len = encode_sample(raw_audio)
            text = entries[pos].get('whisper', '')
            buf.append((h_trimmed, mel_len, text))
            n_processed += 1
        except Exception as e:
            errors += 1
            print(f'  [ERR] idx={stream_idx}: {e}')
            if errors > 50:
                print('  Too many errors, stopping this dataset.')
                break
            continue

        # Flush completed shard
        if len(buf) >= SS:
            fsize = flush_shard(buf, lang_dir, shard_idx)
            lang_bytes += fsize
            total_bytes += fsize
            shard_idx += 1
            buf.clear()
            pbar.set_postfix_str(
                f'{n_processed}/{len(entries)} | shard={shard_idx} | '
                f'{lang_bytes/1e6:.0f}MB')

    pbar.close()

    # Flush remaining buffer
    if buf:
        fsize = flush_shard(buf, lang_dir, shard_idx)
        lang_bytes += fsize
        total_bytes += fsize
        shard_idx += 1

    n_shards_done = len([f for f in os.listdir(lang_dir) if f.endswith('.npz')])
    print(f'  {safe}: {n_shards_done} shards, {lang_bytes/1e6:.0f} MB, '
          f'{errors} errors')
    Path(done_flag).touch()

print(f'\n✅ Teacher cache complete — {total_bytes/1e9:.2f} GB total')

del t_model, enc
flush()

## Distillation  

In [ ]:
import gc, os, math, time, json, glob
# Free leftovers from the pseudo-labelling / encoder-cache cells.
for _v in ['t_model', 'enc', 'feat_ext', 'processor', 'proc',
           'model', 'whisper_model', 'feature_extractor', 'student', 'optimizer']:
    if _v in dir():
        try: exec(f'del {_v}')
        except Exception: pass
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.synchronize()
    torch.cuda.ipc_collect()

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
from transformers import MambaForCausalLM, MambaConfig
import sentencepiece as spm

device = torch.device('cuda')

# Session time budget
SESSION_HOURS = 8.0
CKPT_EVERY_MINUTES = 30      # rolling save cadence during training
SESSION_START = time.monotonic()
SESSION_DEADLINE = SESSION_START + SESSION_HOURS * 3600

def time_left_seconds():
    return max(0.0, SESSION_DEADLINE - time.monotonic())

def fmt_hms(seconds):
    seconds = int(seconds)
    return f'{seconds // 3600:d}h{(seconds % 3600) // 60:02d}m{seconds % 60:02d}s'

print(f'Session budget: {SESSION_HOURS}h, ckpt every {CKPT_EVERY_MINUTES} min')


# Create checkpoint directory:
# Use absolute C.ckpt_dir, not a relative "checkpoints/" which resolves
# to the kernel CWD and silently diverges from where checkpoints actually live.
os.makedirs(C.ckpt_dir, exist_ok=True)

# ── Weights & Biases setup ──────────────────────────────────────
# Logs per-step train metrics + per-epoch eval (val loss + WER) to wandb.
# Resume across Kaggle sessions: the run_id is saved to disk and reused so
# you get one continuous run across all your training sessions.
WANDB_PROJECT = "edge-asr-distillation"   # ← edit if you want a different project
WANDB_RUN_FILE = os.path.join(C.ckpt_dir, 'wandb_run.json')

# Install if missing (Kaggle base image usually has it, but be safe)
try:
    import wandb
except ImportError:
    os.system("pip install -q wandb")
    import wandb

# Pull API key from Kaggle Secrets; fall back to offline mode if absent.
# To set it: Kaggle notebook  : Add-ons  : Secrets  : add WANDB_API_KEY
if "WANDB_API_KEY" not in os.environ:
    try:
        from kaggle_secrets import UserSecretsClient
        os.environ["WANDB_API_KEY"] = UserSecretsClient().get_secret("wandb_api_key")
        print("WANDB_API_KEY loaded from Kaggle Secrets")
    except Exception as e:
        print(f"No WANDB_API_KEY found ({type(e).__name__}). Logging in OFFLINE mode.")
        print(f"To enable online sync: add WANDB_API_KEY in Add-ons  : Secrets, then rerun.")
        print(f"Or sync afterwards with:  wandb sync /kaggle/working/wandb/<run_dir>")

# Resume an existing wandb run if we've trained before. Check the writable
# dir first (this-session continuation), then the read-only baseline dataset
# (cross-dataset-version continuation). This keeps every Kaggle session's
# losses going into ONE continuous wandb run.
prior_run_id = None
_wandb_run_candidates = [WANDB_RUN_FILE]
if getattr(C, 'ckpt_load_dir', None):
    _wandb_run_candidates.append(os.path.join(C.ckpt_load_dir, 'wandb_run.json'))

for _candidate in _wandb_run_candidates:
    if os.path.exists(_candidate):
        try:
            prior_run_id = json.load(open(_candidate))["run_id"]
            print(f"  ↻ Resuming wandb run {prior_run_id} (from {_candidate})")
            break
        except Exception:
            continue

wandb_mode = "online" if "WANDB_API_KEY" in os.environ else "offline"
wandb_run = wandb.init(
    project=WANDB_PROJECT,
    id=prior_run_id,
    resume="allow",
    mode=wandb_mode,
    config={k: v for k, v in vars(C).items()
            if not k.startswith("_") and isinstance(v, (int, float, str, bool, list, tuple))},
    settings=wandb.Settings(start_method="thread"),
)
if prior_run_id is None:
    json.dump({"run_id": wandb_run.id}, open(WANDB_RUN_FILE, "w"))
    print(f"Started new wandb run: {wandb_run.id} (mode={wandb_mode})")
else:
    print(f"Continuing wandb run: {wandb_run.id} (mode={wandb_mode})")
print(f"  URL: {wandb_run.get_url() or '(offline — sync after training)'}")


# ── Local JSONL loss logger (disk-side backup of wandb) ─────────
# Every metric logged to wandb is also appended here as a single JSON line.
# Append-only, line-buffered, fsync'd on flush — so even if Kaggle kills the
# kernel mid-write only the last line is at risk and previous history is safe.
class LossLogger:
    """Thin append-only JSONL writer keyed by wandb run + global wall clock."""
    def __init__(self, path, run_id):
        self.path = path
        self.run_id = run_id
        os.makedirs(os.path.dirname(path), exist_ok=True)
        # Open in append mode so resumes preserve history.
        # line_buffering=True flushes on every '\n' (each record is one line).
        self._fh = open(path, 'a', buffering=1, encoding='utf-8')
        n_existing = 0
        if os.path.exists(path):
            with open(path, 'r', encoding='utf-8') as f:
                for _ in f:
                    n_existing += 1
        self._n_records = n_existing
        # Header record on session start — lets you separate sessions in post-hoc analysis.
        self._fh.write(json.dumps({
            'type': 'session_start',
            'ts': time.time(),
            'run_id': run_id,
            'session_start_iso': time.strftime('%Y-%m-%dT%H:%M:%S'),
        }) + '\n')

    def log(self, record_type, **fields):
        rec = {'type': record_type, 'ts': time.time(), 'run_id': self.run_id}
        rec.update(fields)
        self._fh.write(json.dumps(rec, default=str) + '\n')
        self._n_records += 1

    def close(self):
        try:
            self._fh.flush()
            os.fsync(self._fh.fileno())
        except Exception:
            pass
        self._fh.close()

loss_log = LossLogger(C.loss_log_path, wandb_run.id)
print(f"Loss log: {C.loss_log_path} ({loss_log._n_records - 1} prior records)")

# ── T4 16 GB overrides ──────────────────────────────────────────
# Reduce per-GPU batch + max audio length to fit a Mamba-130M + CNN +
# heads + grad-ckpt forward/backward into a single T4. Effective batch is
# kept at 32 via gradient accumulation. All changes printed below so the
# active config is visible from one place.
_T4_OVERRIDES = dict(
    fr_bs   = 4,    # frozen-stage micro-batch
    fr_ga   = 8,    # frozen-stage grad-accum (effective bs = 4*8 = 32)
    un_bs   = 2,    # unfrozen micro-batch (was 1 — too noisy for DP)
    un_ga   = 16,   # unfrozen grad-accum (effective bs = 2*16 = 32)
    max_s1  = 6.0,  # max audio seconds during frozen stage
    max_s2  = 8.0,  # max audio seconds during unfrozen stage
)
print('T4 overrides applied:')
for _k, _v in _T4_OVERRIDES.items():
    _old = getattr(C, _k)
    setattr(C, _k, _v)
    print(f'  C.{_k:8s} {_old!r:>8}  ->  {_v!r}')
print(f'  effective batch (frozen)   = {C.fr_bs * C.fr_ga}')
print(f'  effective batch (unfrozen) = {C.un_bs * C.un_ga}')


# Decode + WER helpers (for validation logging)
# Lightweight install of jiwer for clean WER. Falls back to a manual
# Levenshtein implementation if jiwer can't be installed.
try:
    from jiwer import wer as _jiwer_wer
    _HAS_JIWER = True
except ImportError:
    try:
        os.system("pip install -q jiwer")
        from jiwer import wer as _jiwer_wer
        _HAS_JIWER = True
    except Exception:
        _HAS_JIWER = False
        print("    jiwer unavailable, falling back to manual WER computation")


def greedy_ctc_decode(ctc_logits, sp_model, blank_id, lengths=None):
    """Greedy CTC decode: argmax  : collapse repeats  : drop blanks  : detokenize."""
    preds = ctc_logits.argmax(dim=-1)# [B, T]
    out = []
    for i, p in enumerate(preds):
        seq = p[:lengths[i]].tolist() if lengths is not None else p.tolist()
        # Collapse consecutive duplicates, drop blanks
        cleaned, prev = [], -1
        for tok in seq:
            if tok != prev and tok != blank_id:
                cleaned.append(tok)
            prev = tok
        out.append(sp_model.DecodeIds(cleaned))
    return out


def _manual_wer(refs, hyps):
    """Word-level Levenshtein distance, summed over the batch / total ref words."""
    def edit_dist(r, h):
        # Standard DP; r, h are lists of words
        if not r:
            return len(h)
        if not h:
            return len(r)
        prev = list(range(len(h) + 1))
        for i, rw in enumerate(r, 1):
            curr = [i] + [0] * len(h)
            for j, hw in enumerate(h, 1):
                cost = 0 if rw == hw else 1
                curr[j] = min(curr[j-1] + 1,           # insertion
                              prev[j] + 1,             # deletion
                              prev[j-1] + cost)        # substitution / match
            prev = curr
        return prev[-1]

    total_words = total_err = 0
    for r, h in zip(refs, hyps):
        rw, hw = r.split(), h.split()
        total_words += len(rw)
        total_err += edit_dist(rw, hw)
    return total_err / max(total_words, 1)


def compute_wer(refs, hyps):
    if not refs:
        return 0.0
    if _HAS_JIWER:
        # jiwer raises on empty hyp lists; protect just in case
        try:
            return float(_jiwer_wer(refs, hyps))
        except Exception:
            return _manual_wer(refs, hyps)
    return _manual_wer(refs, hyps)

# ── 2. Load SentencePiece tokenizer ─────────────────────────────
sp = spm.SentencePieceProcessor()
sp.Load(C.spm_prefix + '.model')
BLANK = sp.GetPieceSize()  # CTC blank = last index
print(f'Tokenizer: {sp.GetPieceSize()} tokens + blank={BLANK}, vocab_size={C.vocab_size}')


# ── 3. Dataset (lazy-loaded int8 ragged shards) ────────────────
class KDDataset(Dataset):
    def __init__(self, cache_dir, sp_model):
        self.sp = sp_model
        self.index = []  # (shard_path, local_idx)

        shard_paths = []
        for lang_dir in sorted(os.listdir(cache_dir)):
            lang_path = os.path.join(cache_dir, lang_dir)
            if not os.path.isdir(lang_path):
                continue
            shards = sorted(f for f in os.listdir(lang_path) if f.endswith('.npz'))
            for sf in shards:
                shard_paths.append(os.path.join(lang_path, sf))
            print(f'  {lang_dir}: {len(shards)} shards')

        print(f'Indexing {len(shard_paths)} shards...')
        for shard in tqdm(shard_paths, desc='Indexing', unit='shard'):
            data = np.load(shard, allow_pickle=True)
            n = len(data['scales'])
            for j in range(n):
                self.index.append((shard, j))
            del data
        print(f'Indexed {len(self.index)} samples')

        self._cache, self._cache_order = {}, []
        self._max_cache = 4   # keep cache small in RAM

    def _get_shard(self, shard_path):
        if shard_path not in self._cache:
            if len(self._cache) >= self._max_cache:
                oldest = self._cache_order.pop(0)
                del self._cache[oldest]
            data = np.load(shard_path, allow_pickle=True)
            self._cache[shard_path] = {
                'hidden_cat': data['hidden_cat'],
                'offsets': data['offsets'],
                'scales': data['scales'],
                'zeros': data['zeros'],
                'mel_lens': data['mel_lens'],
                'texts': data['texts'],
            }
            self._cache_order.append(shard_path)
        return self._cache[shard_path]

    def __len__(self):
        return len(self.index)

    def __getitem__(self, idx):
        shard_path, local_idx = self.index[idx]
        data = self._get_shard(shard_path)
        s, e = data['offsets'][local_idx], data['offsets'][local_idx + 1]
        h_i8 = data['hidden_cat'][s:e].astype(np.float32)
        h_f16 = ((h_i8 - data['zeros'][local_idx]) * data['scales'][local_idx]).astype(np.float16)
        text = str(data['texts'][local_idx])
        mel_len = int(data['mel_lens'][local_idx])
        token_ids = self.sp.EncodeAsIds(text)
        return {
            'teacher_h': torch.from_numpy(h_f16),
            'mel_len': mel_len,
            'token_ids': torch.tensor(token_ids, dtype=torch.long),
            'text': text,
        }


# ── Shard-bucketed batch sampler ─────────────────────────────────
# Random global sampling forces every batch to potentially hit 4 different
# shards on a slow FUSE input mount — `np.load` per shard dominates step
# time. This sampler groups samples by shard so all samples in a batch
# come from one shard; combined with KDDataset's LRU cache, each shard is
# loaded once per epoch instead of thousands of times.
#
# Randomness preserved: shard order is shuffled per epoch, and samples
# within each shard are shuffled too.
class ShardBucketBatchSampler:
    def __init__(self, dataset, batch_size, generator=None, drop_last=True, shuffle=True):
        """Yields batches of indices where every index in a batch maps
        to the same underlying shard.

        dataset: KDDataset or a Subset wrapping one. Must expose an `index`
                 attribute (list of (shard_path, local_idx) tuples) on the
                 underlying KDDataset.
        """
        self.batch_size = batch_size
        self.generator = generator
        self.drop_last = drop_last
        self.shuffle = shuffle

        from torch.utils.data import Subset
        # Walk through any chain of Subset wrappers (e.g. random_split
        # then a Subset on top, as the smoke cell does) until we reach
        # the underlying KDDataset, which is the only thing with an
        # `.index` attribute. `idx_chain[pos]` maps our position `pos`
        # back to the base-dataset row.
        def _resolve(d):
            if isinstance(d, Subset):
                base, inner = _resolve(d.dataset)
                # Compose: our index list, applied to the inner chain.
                return base, [inner[i] for i in d.indices]
            return d, list(range(len(d)))

        base_dataset, idx_chain = _resolve(dataset)
        shard_to_positions = {}
        for pos, real_i in enumerate(idx_chain):
            shard_path, _ = base_dataset.index[real_i]
            shard_to_positions.setdefault(shard_path, []).append(pos)

        # Stable list of buckets; we'll shuffle indexes into it per epoch.
        self.shard_buckets = list(shard_to_positions.values())

        if self.drop_last:
            self._n_batches = sum(len(b) // self.batch_size
                                  for b in self.shard_buckets)
        else:
            self._n_batches = sum((len(b) + self.batch_size - 1) // self.batch_size
                                  for b in self.shard_buckets)

    def __len__(self):
        return self._n_batches

    def __iter__(self):
        n_buckets = len(self.shard_buckets)
        # Determine shard iteration order.
        if not self.shuffle:
            shard_order = list(range(n_buckets))
        elif self.generator is not None:
            shard_order = torch.randperm(n_buckets, generator=self.generator).tolist()
        else:
            import random
            shard_order = list(range(n_buckets))
            random.shuffle(shard_order)

        for s_idx in shard_order:
            bucket = self.shard_buckets[s_idx]
            # Determine within-shard sample order.
            if not self.shuffle:
                pass  # use natural order
            elif self.generator is not None:
                perm = torch.randperm(len(bucket), generator=self.generator).tolist()
                bucket = [bucket[i] for i in perm]
            else:
                import random
                bucket = bucket.copy()
                random.shuffle(bucket)

            for i in range(0, len(bucket), self.batch_size):
                batch = bucket[i:i + self.batch_size]
                if len(batch) < self.batch_size and self.drop_last:
                    continue
                yield batch


# ── Persistent train/val/test splits ─────────────────────────────
# Kaggle sessions are capped at 9-12 hours. A multi-stage distillation
# run will span many sessions, so the train/val/test split MUST be
# stable across them — otherwise samples that were training data last
# session become val data this session, leaking the model's training
# signal into the metric. These helpers persist split indices to a JSON
# file under C.ckpt_dir, with a dataset-state fingerprint to refuse
# silently using stale splits if shards were added or removed.
def _dataset_fingerprint(ds):
    """Short hash of dataset state. Changes if shards are added, removed,
    or reordered. Sampled across the dataset, not just the head."""
    import hashlib
    h = hashlib.sha256()
    n = len(ds)
    h.update(f'len={n}'.encode())
    step = max(1, n // 100)
    for i in range(0, n, step):
        shard_path, local_idx = ds.index[i]
        # Use last two path components (language_dir/shard_file) so the
        # fingerprint survives moving the cache to a different mount.
        parts = shard_path.replace('\\', '/').rsplit('/', 2)[-2:]
        h.update(f'{"/".join(parts)}:{local_idx}'.encode())
    return h.hexdigest()[:16]


def get_or_create_splits(ds, val_size, test_size, seed, splits_path):
    """Persistent train/val/test Subsets.

    First call: generates indices via torch.randperm(seed), saves them and
        a dataset fingerprint to splits_path, returns three Subsets.
    Later calls: loads from splits_path, verifies fingerprint, returns
        Subsets reconstructed from the saved indices.

    Refuses (raises RuntimeError) if the fingerprint mismatches — this
    prevents silently using a stale split if shards were added/removed
    between sessions.
    """
    from torch.utils.data import Subset

    fingerprint = _dataset_fingerprint(ds)

    if os.path.exists(splits_path):
        with open(splits_path) as f:
            saved = json.load(f)
        if saved.get('fingerprint') != fingerprint:
            raise RuntimeError(
                f'Splits in {splits_path} were created for a different '
                f'dataset state:\n'
                f'  saved fingerprint   = {saved.get("fingerprint")}\n'
                f'  current fingerprint = {fingerprint}\n'
                f'  saved dataset_len   = {saved.get("dataset_len")}\n'
                f'  current dataset_len = {len(ds)}\n'
                f'Either restore the cache to the original state, or '
                f'delete {splits_path} to regenerate (NOTE: this invalidates '
                f'any test results from previous methods using the old splits).'
            )
        print(f'Loaded persisted splits from {splits_path}')
        print(f'  seed={saved["seed"]}, fingerprint={fingerprint}, '
              f'train/val/test = {len(saved["train_indices"]):,} / '
              f'{len(saved["val_indices"]):,} / {len(saved["test_indices"]):,}')
        return (
            Subset(ds, saved['train_indices']),
            Subset(ds, saved['val_indices']),
            Subset(ds, saved['test_indices']),
        )

    # First time: generate, save, return.
    n = len(ds)
    if val_size + test_size >= n:
        raise ValueError(f'val_size + test_size = {val_size + test_size} '
                         f'>= dataset size {n}')
    train_size = n - val_size - test_size

    gen = torch.Generator().manual_seed(seed)
    perm = torch.randperm(n, generator=gen).tolist()
    train_idx = perm[:train_size]
    val_idx = perm[train_size:train_size + val_size]
    test_idx = perm[train_size + val_size:]

    os.makedirs(os.path.dirname(splits_path) or '.', exist_ok=True)
    with open(splits_path, 'w') as f:
        json.dump({
            'fingerprint':    fingerprint,
            'seed':           seed,
            'dataset_len':    n,
            'val_size':       val_size,
            'test_size':      test_size,
            'train_indices':  train_idx,
            'val_indices':    val_idx,
            'test_indices':   test_idx,
        }, f)
    print(f'Created splits at {splits_path}: '
          f'train={len(train_idx):,}, val={len(val_idx):,}, '
          f'test={len(test_idx):,} (seed={seed})')
    from torch.utils.data import Subset
    return Subset(ds, train_idx), Subset(ds, val_idx), Subset(ds, test_idx)


def collate_kd(batch, max_enc_frames=None):
    max_t = max(b['teacher_h'].shape[0] for b in batch)
    if max_enc_frames:
        max_t = min(max_t, max_enc_frames)
    D = batch[0]['teacher_h'].shape[1]

    teacher_h = torch.zeros(len(batch), max_t, D, dtype=torch.float16)
    teacher_mask = torch.zeros(len(batch), max_t, dtype=torch.bool)
    mel_lens = []
    for i, b in enumerate(batch):
        t = min(b['teacher_h'].shape[0], max_t)
        teacher_h[i, :t] = b['teacher_h'][:t]
        teacher_mask[i, :t] = True
        mel_lens.append(b['mel_len'])

    max_tok = max(len(b['token_ids']) for b in batch)
    tokens = torch.full((len(batch), max_tok), BLANK, dtype=torch.long)
    tok_lens = []
    for i, b in enumerate(batch):
        tl = len(b['token_ids'])
        tokens[i, :tl] = b['token_ids']
        tok_lens.append(tl)

    return {
        'teacher_h': teacher_h,
        'teacher_mask': teacher_mask,
        'mel_lens': torch.tensor(mel_lens, dtype=torch.long),
        'tokens': tokens,
        'tok_lens': torch.tensor(tok_lens, dtype=torch.long),
        'texts': [b['text'] for b in batch],   # for WER (eval only)
    }


# ── 4. Student Model ───────────────────────────────────────────
class StudentASRv2(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config

        self.train_proj = nn.Sequential(
            nn.Linear(config.teacher_d, config.mamba_d),
            nn.GELU(),
            nn.LayerNorm(config.mamba_d),
        )

        self.cnn = nn.Sequential(
            nn.Conv1d(config.n_mels, config.cnn_ch, config.cnn_ks,
                      stride=2, padding=config.cnn_ks // 2),
            nn.GELU(),
            nn.Conv1d(config.cnn_ch, config.cnn_ch, config.cnn_ks,
                      stride=2, padding=config.cnn_ks // 2),
            nn.GELU(),
        )
        self.cnn_proj = nn.Linear(config.cnn_ch, config.mamba_d)

        # Load Mamba in fp32 AND run its selective-scan in fp32 (see forward):
        # autocast would otherwise cast the SSM activations to fp16, and the
        # scan recurrence overflows fp16 over long sequences -> inf/NaN. Keeping
        # only the weights in fp32 is not enough; the compute must be fp32 too.
        self.mamba = MambaForCausalLM.from_pretrained(
            config.mamba_pretrained, torch_dtype=torch.float32,
        ).backbone

        mamba_d_actual = self.mamba.config.hidden_size
        if mamba_d_actual != config.mamba_d:
            print(f'Mamba hidden={mamba_d_actual}, adjusting config')
            config.mamba_d = mamba_d_actual
            self.train_proj = nn.Sequential(
                nn.Linear(config.teacher_d, mamba_d_actual),
                nn.GELU(),
                nn.LayerNorm(mamba_d_actual),
            )
            self.cnn_proj = nn.Linear(config.cnn_ch, mamba_d_actual)

        self.kl_head = nn.Linear(config.mamba_d, config.teacher_d)
        self.ctc_head = nn.Linear(config.mamba_d, config.vocab_size)

        # Regularization. nn.Dropout has no parameters, so adding this
        # does NOT change state_dict keys — existing checkpoints load fine.
        # Set config.dropout=0.0 to disable.
        self.dropout = nn.Dropout(getattr(config, 'dropout', 0.0))

    def enable_grad_ckpt(self):
        """
        Per-block activation checkpointing. Trades ~10-15% extra compute for
        roughly 4-8x less activation memory through the 24 Mamba layers.
        DataParallel-safe.

        Why class-level patching: the previous implementation wrapped each
        block's `forward` with a closure capturing the bound method
        `orig = block.forward`. The bound method holds a hard reference to
        the cuda:0 weights — so when DataParallel replicated the model to
        cuda:1, the new replica's `block.forward = wrap`, but the closure
        inside `wrap` still pointed at the cuda:0 weights → device mismatch.

        This version patches the BLOCK CLASS once. Every instance (incl.
        DataParallel replicas) gets the same wrapped forward, and `self`
        inside resolves to the per-replica instance on the right GPU.
        Idempotent: safe to call multiple times.
        """
        from torch.utils.checkpoint import checkpoint
        if not self.mamba.layers:
            return

        block_cls = type(self.mamba.layers[0])
        if getattr(block_cls, '_grad_ckpt_patched', False):
            print(f'Gradient checkpointing already enabled on '
                  f'{block_cls.__name__}')
            return

        original_forward = block_cls.forward  # unbound class function

        def ckpt_forward(self, hidden_states, *args, **kwargs):
            # No checkpoint at eval time — saves the recompute overhead.
            if not self.training:
                return original_forward(self, hidden_states, *args, **kwargs)
            # `self` and `original_forward` are correctly scoped: when
            # DataParallel calls this on a replica, `self` is the replica
            # (on cuda:N), and `original_forward(self, ...)` dispatches to
            # that replica's parameters. No closure to cuda:0.
            return checkpoint(
                lambda h: original_forward(self, h, *args, **kwargs),
                hidden_states,
                use_reentrant=False,
            )

        block_cls.forward = ckpt_forward
        block_cls._grad_ckpt_patched = True
        print(f'Gradient checkpointing enabled on {len(self.mamba.layers)} '
              f'{block_cls.__name__} blocks (DataParallel-safe)')

    def forward(self, teacher_h, teacher_mask):
        # Keep teacher_h in fp16: autocast will downcast train_proj outputs anyway,
        # and we save 2x peak memory vs an explicit .float() upcast.
        x = self.train_proj(teacher_h)
        # fp32 island: the Mamba selective-scan overflows fp16 over long
        # sequences. Weights are already fp32; force fp32 activations too.
        with torch.amp.autocast('cuda', enabled=False):
            h = self.mamba(inputs_embeds=x.float()).last_hidden_state
        h = self.dropout(h)  # no-op in eval() mode
        return self.kl_head(h), self.ctc_head(h)

    def forward_mel(self, mel):
        x = self.cnn(mel)
        x = x.transpose(1, 2)
        x = self.cnn_proj(x)
        with torch.amp.autocast('cuda', enabled=False):
            h = self.mamba(inputs_embeds=x.float()).last_hidden_state
        h = self.dropout(h)
        return self.ctc_head(h)


# ── 5. Loss Functions ──────────────────────────────────────────
# def kd_loss_chunked(student_kl, teacher_h, mask, temperature=2.0, chunk=128):
#     """
#     Frame-level KL between student logits and teacher hidden states, computed
#     over the feature dim. Original implementation built [B, T, 1280] fp32
#     softmax tensors twice (~80 MB at B=16, T=500). We chunk along T so peak
#     stays below ~10 MB regardless of sequence length.
#     """
#     B, T, D = student_kl.shape
#     total = student_kl.new_zeros(())
#     n_valid = mask.float().sum().clamp(min=1)
#     for s in range(0, T, chunk):
#         e = min(s + chunk, T)
#         s_log = F.log_softmax(student_kl[:, s:e].float() / temperature, dim=-1)
#         with torch.no_grad():
#             t_prob = F.softmax(teacher_h[:, s:e].float() / temperature, dim=-1)
#         kl = F.kl_div(s_log, t_prob, reduction='none').sum(-1)  # [B, e-s]
#         total = total + (kl * mask[:, s:e].float()).sum()
#     return (total / n_valid) * (temperature ** 2)

def kd_feature_loss(student_feat, teacher_feat, mask, chunk=128):
    """
    Frame-level feature distillation via 1 - cosine_similarity.

    Why not KL: teacher_feat is the cached Whisper ENCODER hidden state
    [B, T, 1280] — raw activations, not logits. Softmaxing them over the
    feature dim treats them as a distribution, which they aren't, and the
    result is near-uniform  : KL ≈ 0 (the bug). The student's kl_head is a
    Linear(mamba_d  : teacher_d) projection: we want it to *match* the
    teacher's features, which is exactly what cosine sim measures.

    Scale-invariant (robust to the int8 quantize/dequantize round-trip in
    the cached shards) and gives a strong gradient at init (uncorrelated
    random projections  : cosine ≈ 0  : loss ≈ 1).

    Args:
        student_feat, teacher_feat: [batch_size, time_steps/seq, features_dimension]
        mask: [B, T] bool, True = valid frame
        chunk: T-axis chunk size for memory control
    Returns:
        scalar in [0, 2]; expect 0.8–1.0 at step 1, dropping to 0.1–0.3.
    """
    B, T, D = student_feat.shape
    total = student_feat.new_zeros((), dtype=torch.float32)
    mask_f = mask.float()
    n_valid = mask_f.sum().clamp(min=1)
    for s in range(0, T, chunk):
        e = min(s + chunk, T)
        # fp32 dot product avoids fp16 underflow when features are small.
        cos = F.cosine_similarity(
            student_feat[:, s:e].float(),
            teacher_feat[:, s:e].float(),
            dim=-1,
        )  # [B, e-s]
        total = total + ((1.0 - cos) * mask_f[:, s:e]).sum()
    return total / n_valid


def ctc_loss_fn(ctc_logits, tokens, enc_lens, tok_lens):
    log_probs = F.log_softmax(ctc_logits.float(), dim=-1).transpose(0, 1)
    return F.ctc_loss(
        log_probs, tokens, enc_lens, tok_lens,
        blank=BLANK, zero_infinity=True,
    )

### Smoke test (run first — gates the training cell below)

Runs ~50 gradient steps on a tiny subset to validate the fast-path Mamba kernels, memory budget, and loss trajectory before committing to a multi-hour run. Sets `_SMOKE_PASSED = True` on success; the training cell refuses to start otherwise.


In [ ]:
# 9. SMOKE TEST — gates the full training cell below. Sets _SMOKE_PASSED.
# Runs ~50 gradient steps on a tiny subset to verify the Mamba fast path is
# active, dataloader keeps up with compute, and loss is moving in the right
# direction before committing to a multi-hour run.
from torch.utils.data import Subset, DataLoader
import statistics, time

N_SMOKE = 50
SMOKE_BS = C.fr_bs

# ── Build (mirrors the training cell so the smoke reflects real conditions)
print('Loading dataset...')
ds = KDDataset(C.cache_dir, sp)
flush()

# Persistent splits — same as training cell uses. First run creates
# splits.json under C.ckpt_dir; later sessions load the saved indices
# so the test set never leaks into train.
SPLITS_PATH = os.path.join(C.ckpt_dir, 'splits.json')
_val_size = max(50, min(2000, len(ds) // 50))
_test_size = _val_size
train_ds, val_ds, test_ds = get_or_create_splits(
    ds, val_size=_val_size, test_size=_test_size,
    seed=C.seed, splits_path=SPLITS_PATH,
)
print(f'  train: {len(train_ds):,} samples')
print(f'  val:   {len(val_ds):,} samples (held out, deterministic)')
print(f'  test:  {len(test_ds):,} samples (held out for final eval — '
      f'NOT touched during training)')

print('\nBuilding student model...')
student = StudentASRv2(C).to(device)
student.enable_grad_ckpt()

n_params = sum(p.numel() for p in student.parameters())
n_trainable = sum(p.numel() for p in student.parameters() if p.requires_grad)
print(f'Student: {n_params/1e6:.1f}M params ({n_trainable/1e6:.1f}M trainable)')
flush()

# Frozen-stage configuration (mirrors the real frozen stage of training).
for p in student.mamba.parameters():
    p.requires_grad = False
trainable = sum(p.numel() for p in student.parameters() if p.requires_grad)
print(f"Smoke test: {N_SMOKE} steps, bs={SMOKE_BS}, "
      f"{trainable/1e6:.1f}M trainable (Mamba frozen)")

# Small subset + fresh optimizer/scaler so the smoke is self-contained.
# Need enough samples per shard for the bucket sampler to yield batches.
# ~12 samples/shard at 165 shards = 2000 samples -> ~3 batches per shard
# at bs=4. With 50 smoke steps that's ~16 shard transitions, a fair test.
smoke_size = min(2000, len(train_ds))
smoke_ds = Subset(train_ds, list(range(smoke_size)))
opt = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, student.parameters()),
    lr=C.fr_lr, weight_decay=0.01,
)
scaler = torch.amp.GradScaler('cuda')

max_enc_frames = int(C.max_s1 * 100 / 2)

# IMPORTANT: use workers here. The earlier num_workers=0 made the smoke
# misleading — the FUSE-backed np.load of each shard dominated step time
# (~8 s) and hid the fact that compute itself is fast (~0.5 s). Real
# training uses workers anyway, so the smoke should too.
# Shard-bucketed batch sampling: every batch comes from one shard, so
# the LRU cache hits ~100% within a bucket. Combined with workers
# prefetching, data-load should drop from ~3s to <0.3s per step.
smoke_sampler = ShardBucketBatchSampler(
    smoke_ds, batch_size=SMOKE_BS,
    generator=torch.Generator().manual_seed(C.seed),
    drop_last=True,
)
loader = DataLoader(
    smoke_ds, batch_sampler=smoke_sampler,
    num_workers=C.workers, persistent_workers=(C.workers > 0),
    pin_memory=True, prefetch_factor=2 if C.workers > 0 else None,
    collate_fn=lambda b: collate_kd(b, max_enc_frames=max_enc_frames),
)
_bucket_sizes = sorted(len(b) for b in smoke_sampler.shard_buckets)
_nonempty = [s for s in _bucket_sizes if s >= SMOKE_BS]
print(f'  ShardBucketBatchSampler: {len(smoke_sampler)} batches '
      f'across {len(smoke_sampler.shard_buckets)} shards '
      f'({len(_nonempty)} of which have >= {SMOKE_BS} samples)')
print(f'  bucket-size percentiles: '
      f'min={_bucket_sizes[0]}, p25={_bucket_sizes[len(_bucket_sizes)//4]}, '
      f'p50={_bucket_sizes[len(_bucket_sizes)//2]}, '
      f'p75={_bucket_sizes[3*len(_bucket_sizes)//4]}, max={_bucket_sizes[-1]}')

student.train()
step_times, load_times, compute_times = [], [], []
losses_kl, losses_ctc = [], []
torch.cuda.reset_peak_memory_stats()

print(f"\n{'step':>4} {'loss':>7} {'l_kl':>8} {'l_ctc':>7} "
      f"{'mem_GB':>7} {'load':>5} {'cmp':>5} {'tot':>5}")
print("-" * 60)

# Hand-roll the iterator so we can time data-load (next(it)) vs compute
# separately. dataloader returns the next batch from workers' queue.
it = iter(loader)
t_prev = time.monotonic()
for step in range(N_SMOKE):
    # ── 1. Data load (worker queue dequeue + collate already done in workers)
    t_load_start = time.monotonic()
    try:
        batch = next(it)
    except StopIteration:
        break
    t_load_done = time.monotonic()

    teacher_h    = batch['teacher_h'].to(device, non_blocking=True)
    teacher_mask = batch['teacher_mask'].to(device, non_blocking=True)
    tokens       = batch['tokens'].to(device, non_blocking=True)
    tok_lens     = batch['tok_lens'].to(device, non_blocking=True)
    mel_lens     = batch['mel_lens'].to(device, non_blocking=True)
    enc_lens     = teacher_mask.sum(dim=1).long().clamp(min=1)  # true valid-frame count

    # ── 2. Compute
    t_compute_start = time.monotonic()
    with torch.amp.autocast('cuda', dtype=torch.float16):
        kl_logits, ctc_logits = student(teacher_h, teacher_mask)
        loss_kl  = kd_feature_loss(kl_logits, teacher_h, teacher_mask)
        loss_ctc = ctc_loss_fn(ctc_logits, tokens, enc_lens, tok_lens)
        loss     = C.a_kl * loss_kl + C.a_ctc * loss_ctc

    if not torch.isfinite(loss):
        _SMOKE_PASSED = False
        raise RuntimeError(
            f"Non-finite loss at smoke step {step}: "
            f"loss={loss.item()} kl={loss_kl.item()} ctc={loss_ctc.item()}. "
            "Mamba SSM likely overflowed fp16 — confirm the fp32 island is in forward()."
        )

    scaler.scale(loss).backward()
    scaler.step(opt)
    scaler.update()
    opt.zero_grad(set_to_none=True)
    torch.cuda.synchronize()
    t_compute_done = time.monotonic()

    load_t = t_load_done - t_load_start
    compute_t = t_compute_done - t_compute_start
    total_t = t_compute_done - t_prev
    t_prev = t_compute_done

    load_times.append(load_t)
    compute_times.append(compute_t)
    step_times.append(total_t)
    losses_kl.append(loss_kl.item())
    losses_ctc.append(loss_ctc.item())

    if step < 5 or step % 10 == 0 or step == N_SMOKE - 1:
        mem_gb = torch.cuda.memory_allocated() / 1e9
        print(f"{step:4d} {loss.item():7.3f} {loss_kl.item():8.4f} "
              f"{loss_ctc.item():7.3f} {mem_gb:7.2f} "
              f"{load_t:5.2f} {compute_t:5.2f} {total_t:5.2f}")

# ── Verdict ─────────────────────────────────────────────────────
print("\n" + "=" * 60)
# Skip the first 2 steps for medians — they include worker spin-up
# and first-batch JIT compile.
warm = max(2, len(step_times) // 10)
def med(xs):
    return statistics.median(xs[warm:]) if len(xs) > warm else (xs[-1] if xs else 0.0)

med_step    = med(step_times)
med_load    = med(load_times)
med_compute = med(compute_times)
peak_mem    = torch.cuda.max_memory_allocated() / 1e9
kl_first    = sum(losses_kl[:5]) / min(5, len(losses_kl))
kl_last     = sum(losses_kl[-5:]) / min(5, len(losses_kl))
ctc_first   = sum(losses_ctc[:5]) / min(5, len(losses_ctc))
ctc_last    = sum(losses_ctc[-5:]) / min(5, len(losses_ctc))

print(f"Median sec/step (after warmup): {med_step:.2f}s")
print(f"  - data load: {med_load:.2f}s  ({100*med_load/max(med_step,1e-3):.0f}% of step)")
print(f"  - compute  : {med_compute:.2f}s  ({100*med_compute/max(med_step,1e-3):.0f}% of step)")
print(f"Peak GPU memory:                {peak_mem:.2f} GB")
print(f"loss_kl  first 5  : last 5: {kl_first:.4f}  : {kl_last:.4f}  (Δ {kl_last-kl_first:+.4f})")
print(f"loss_ctc first 5  : last 5: {ctc_first:.3f}  : {ctc_last:.3f}  (Δ {ctc_last-ctc_first:+.3f})")
print()

# Context-aware thresholds. Smoke runs with Mamba FROZEN — only ~10M params
# trainable, no Mamba gradients/optimizer-state, so peak memory should be
# 0.5-2 GB at bs=4. The previous "≥ 3 GB" threshold was calibrated for the
# unfrozen case and produced false alarms.
issues = []
# Step-time: with workers active and compute at ~0.5 s, we should be ≤ 2 s/step.
if med_step > 2.0:
    if med_load > med_compute:
        issues.append(
            f"step {med_step:.1f}s is dataloader-bound "
            f"(load {med_load:.1f}s >> compute {med_compute:.1f}s). "
            f"Try increasing C.workers (currently {C.workers}) or prefetch_factor."
        )
    else:
        issues.append(
            f"step {med_step:.1f}s with compute {med_compute:.1f}s — "
            f"Mamba slow path may be active. Re-run diagnostic_cell."
        )
# Compute floor: fast-path Mamba on T4 bs=4 T=300 frozen ≈ 0.3-0.7s.
if med_compute > 2.0:
    issues.append(
        f"compute {med_compute:.2f}s/step too high for fast-path Mamba "
        "(expected ~0.3-0.7s on T4 bs=4 frozen)"
    )
# Loss trajectory.
if kl_last >= kl_first - 0.005:
    issues.append("loss_kl not decreasing — feature distillation isn't learning")
if kl_first < 0.3:
    issues.append(f"loss_kl starts at {kl_first:.3f} — unexpectedly small, "
                  "check kd_feature_loss call site")
# Memory: just sanity-check we didn't accidentally blow past T4 capacity.
if peak_mem > 14.0:
    issues.append(f"peak mem {peak_mem:.1f} GB — close to T4's 16 GB limit; "
                  "reduce bs or max_s1")

if issues:
    _SMOKE_PASSED = False
    print("  Issues detected:")
    for i in issues:
        print(f"  - {i}")
    print("\nDo NOT proceed to full training until these are resolved.")
    print("(_SMOKE_PASSED = False  — the training cell will refuse to start.)")
else:
    _SMOKE_PASSED = True
    print("✅ All checks pass — _SMOKE_PASSED = True")
    print("   Proceed to the training cell below.")

# Free smoke-only state. Keep `student` itself — the training cell will
# rebuild it, but freeing here would just trigger an immediate realloc.
del opt, scaler, loader, smoke_ds, smoke_sampler, it
torch.cuda.empty_cache()


### Full training (frozen stage + unfrozen stage)

Runs the full multi-epoch distillation. Resumes from `C.ckpt_dir/<stage>_latest.pt` if present, otherwise from the read-only baseline at `C.ckpt_load_dir`. Auto-pauses when the session time budget is exhausted; click *Save Version* and re-run to continue.


In [ ]:
# ── Pre-flight: refuse to start training until the smoke-test cell has run
# and passed. The smoke cell sets _SMOKE_PASSED=True; this guard catches the
# common mistake of running cells top-down on first execution without running
# the smoke test first. Set SKIP_SMOKE_GUARD=True in a prior cell to bypass.
if not globals().get('SKIP_SMOKE_GUARD', False):
    assert globals().get('_SMOKE_PASSED', False), (
        "Run the smoke-test cell FIRST and confirm it passes "
        "(it sets _SMOKE_PASSED=True). To bypass: SKIP_SMOKE_GUARD = True"
    )

# ── 6. Build everything ────────────────────────────────────────
print('Loading dataset...')
ds = KDDataset(C.cache_dir, sp)
flush()

# Train/val/test splits — persisted to splits.json so they stay stable
# across sessions (critical: otherwise a sample that was training data
# last week could be in val/test this week, leaking the signal). The
# test set is held out for final comparison against other methods —
# the training loop never touches it.
SPLITS_PATH = os.path.join(C.ckpt_dir, 'splits.json')
_val_size = max(50, min(2000, len(ds) // 50))   # ~2%, capped at [50, 2000]
_test_size = _val_size                           # symmetric val/test
train_ds, val_ds, test_ds = get_or_create_splits(
    ds, val_size=_val_size, test_size=_test_size,
    seed=C.seed, splits_path=SPLITS_PATH,
)
print(f'  train: {len(train_ds):,} samples')
print(f'  val:   {len(val_ds):,} samples (held out, deterministic)')
print(f'  test:  {len(test_ds):,} samples (held out for final eval — '
      f'never touched by training)')

print('\nBuilding student model...')
student = StudentASRv2(C).to(device)
student.enable_grad_ckpt()

# ── Multi-GPU: wrap in DataParallel when 2+ GPUs are visible.
# DataParallel replicates the model on each GPU and splits the batch along
# dim 0, so the effective per-step batch is `bs * NUM_GPUS`. Works inside a
# notebook without launcher gymnastics (unlike DDP). After wrapping, all
# attribute/method access (e.g. .mamba, .enable_grad_ckpt) must go through
# `student.module`, which the helper below exposes as `student_core`.
if torch.cuda.device_count() > 1:
    student = torch.nn.DataParallel(student)
    print(f'Wrapped student in DataParallel across {torch.cuda.device_count()} GPUs')
student_core = student.module if isinstance(student, torch.nn.DataParallel) else student

n_params = sum(p.numel() for p in student_core.parameters())
n_trainable = sum(p.numel() for p in student_core.parameters() if p.requires_grad)
print(f'Student: {n_params/1e6:.1f}M params ({n_trainable/1e6:.1f}M trainable)')
flush()


# ── 7. Checkpoint helpers ──────────────────────────────────────
def _atomic_save(state, path):
    """Write to .tmp then rename. POSIX rename is atomic, so a kill mid-write
    leaves the previous checkpoint intact instead of corrupting it."""
    # Guard: catch path-argument mistakes (empty / dir / trailing slash) before
    # they bubble up as opaque torch zipfile errors deep in C++.
    if not path or path.endswith(os.sep) or os.path.isdir(path):
        raise ValueError(
            f'_atomic_save got an invalid path: {path!r}. '
            f'Expected a full file path like '
            f'{os.path.join(C.ckpt_dir, "frozen_latest.pt")!r}.'
        )
    # Make sure the parent dir exists (handles the case where the writable
    # dir was deleted between checkpoint writes).
    os.makedirs(os.path.dirname(path) or '.', exist_ok=True)
    tmp = path + '.tmp'
    torch.save(state, tmp)
    os.replace(tmp, path)


def _save_path(stage_name, suffix='latest'):
    """Always writable destination — every save uses this."""
    return os.path.join(C.ckpt_dir, f'{stage_name}_{suffix}.pt')


def _load_candidates(stage_name, suffix='latest'):
    """Ordered list of paths to try when loading.

    Priority:
      1. C.ckpt_dir       — writable. Wins if a prior step in THIS session
                            already saved here (within-session resume).
      2. C.ckpt_load_dir  — read-only. The baseline shipped via the Kaggle
                            input dataset. Wins on a fresh session start.
    """
    fname = f'{stage_name}_{suffix}.pt'
    paths = [os.path.join(C.ckpt_dir, fname)]
    if getattr(C, 'ckpt_load_dir', None) and C.ckpt_load_dir != C.ckpt_dir:
        paths.append(os.path.join(C.ckpt_load_dir, fname))
    return paths


# Back-compat shims so the rest of the code (and external scripts) still work.
def _latest_path(stage_name):  # treated as save path everywhere it's used
    return _save_path(stage_name, 'latest')


def _best_path(stage_name):
    return _save_path(stage_name, 'best')


def _unwrap(model):
    """Return the inner module if `model` is DataParallel, else `model`.
    Use this whenever you would call `.state_dict()` or `.load_state_dict()`
    on the model — checkpoints stay portable between single- and multi-GPU
    sessions because the `module.` prefix never enters the on-disk format.
    """
    return model.module if isinstance(model, torch.nn.DataParallel) else model


def _normalize_state_dict(sd):
    """Strip any `module.` prefix from keys, so the dict matches an
    un-wrapped model. No-op if no keys are prefixed."""
    if not any(k.startswith('module.') for k in sd):
        return sd
    return {(k[len('module.'):] if k.startswith('module.') else k): v
            for k, v in sd.items()}


def save_full_checkpoint(path, *, model, optimizer, scheduler, scaler,
                         stage_name, epoch, batch_idx, global_step, best_loss,
                         best_val_metric=float('inf'),
                         epochs_since_improvement=0,
                         early_stopped=False):
    """Save everything needed to resume training exactly.

    Includes early-stopping state so resumes across sessions correctly
    track epochs-without-improvement.
    """
    state = {
        # Save via _unwrap so the on-disk format never carries the DP
        # `module.` prefix — keeps checkpoints portable across sessions
        # that may or may not use DataParallel.
        'model':        _unwrap(model).state_dict(),
        'optimizer':    optimizer.state_dict(),
        'scheduler':    scheduler.state_dict(),
        'scaler':       scaler.state_dict(),  # preserves fp16 loss scale
        'stage_name':   stage_name,
        'epoch':        epoch,
        'batch_idx':    batch_idx,            # for within-epoch resume
        'global_step':  global_step,
        'best_loss':    best_loss,
        # Early-stopping state (carried across session boundaries).
        'best_val_metric':           best_val_metric,
        'epochs_since_improvement':  epochs_since_improvement,
        'early_stopped':             early_stopped,
        'config':       C,
        'torch_version': torch.__version__,
    }
    _atomic_save(state, path)


def save_best_model_only(path, model, *, epoch, global_step, loss):
    """Save just the model weights — used for the 'best' checkpoint, since
    we never resume from it (it's only for inference/eval)."""
    _atomic_save({
        # See _unwrap rationale in save_full_checkpoint.
        'model': _unwrap(model).state_dict(),
        'epoch': epoch,
        'global_step': global_step,
        'loss': loss,
    }, path)


def try_load_full_checkpoint(path_or_paths):
    """Try a single path or an ordered list of candidates.

    Returns (state_dict, loaded_from_path) on success, (None, None) on failure.
    Each candidate also looks at its .tmp sibling in case a rename was
    interrupted (extremely rare with atomic os.replace).
    """
    if isinstance(path_or_paths, str):
        candidates = [path_or_paths]
    else:
        candidates = list(path_or_paths)

    for path in candidates:
        if not os.path.exists(path):
            continue
        try:
            state = torch.load(path, map_location='cpu', weights_only=False)
            return state, path
        except Exception as e:
            print(f'Could not load {path}: {e}')
            tmp = path + '.tmp'
            if os.path.exists(tmp):
                try:
                    state = torch.load(tmp, map_location='cpu', weights_only=False)
                    return state, tmp
                except Exception:
                    pass
            # Try the next candidate.

    return None, None


# ── 8. Training Loop ───────────────────────────────────────────
@torch.no_grad()
def evaluate(student, val_dataset, max_seconds, bs=2, max_batches=None, a_ctc=None):
    """Run validation: KL + CTC loss and greedy-decode WER.

    Uses fp16 autocast for speed/memory (matches training compute path).
    Capped by `max_batches` to keep eval cheap inside the time budget.

    `a_ctc` overrides C.a_ctc for the composite val/loss, so that during
    the frozen stage (where training uses a_ctc=0) the val/loss tracks the
    same objective the optimizer is actually minimizing — otherwise early
    stopping would compare against a different loss landscape.
    """
    if a_ctc is None:
        a_ctc = C.a_ctc
    student.eval()
    max_enc_frames = int(max_seconds * 100 / 2)
    # Shard-bucketed sampling: keep eval deterministic (no generator).
    val_sampler = ShardBucketBatchSampler(
        val_dataset, batch_size=bs, generator=None,
        drop_last=False, shuffle=False,  # deterministic eval order
    )
    loader = DataLoader(
        val_dataset, batch_sampler=val_sampler,
        num_workers=C.workers, persistent_workers=(C.workers > 0),
        pin_memory=True,
        collate_fn=lambda b: collate_kd(b, max_enc_frames=max_enc_frames),
    )

    total_loss = total_kl = total_ctc = 0.0
    n_batches = 0
    refs, hyps = [], []

    for batch_idx, batch in enumerate(loader):
        if max_batches is not None and batch_idx >= max_batches:
            break

        teacher_h = batch['teacher_h'].to(device, non_blocking=True)
        teacher_mask = batch['teacher_mask'].to(device, non_blocking=True)
        tokens = batch['tokens'].to(device, non_blocking=True)
        tok_lens = batch['tok_lens'].to(device, non_blocking=True)
        mel_lens = batch['mel_lens'].to(device, non_blocking=True)
        # Use the true per-sample valid frame count from teacher_mask. The previous
        # `mel_lens // 2` assumed Whisper's exact 2x encoder stride and could
        # drift from the actual cached/padded length.
        enc_lens = teacher_mask.sum(dim=1).long().clamp(min=1)

        with torch.amp.autocast('cuda', dtype=torch.float16):
            kl_logits, ctc_logits = student(teacher_h, teacher_mask)
            loss_kl = kd_feature_loss(kl_logits, teacher_h, teacher_mask)
            loss_ctc = ctc_loss_fn(ctc_logits, tokens, enc_lens, tok_lens)
            loss = C.a_kl * loss_kl + a_ctc * loss_ctc

        total_loss += loss.item()
        total_kl += loss_kl.item()
        total_ctc += loss_ctc.item()
        n_batches += 1

        # Greedy CTC decode for WER (use fp32 logits for argmax stability).
        decoded = greedy_ctc_decode(
            ctc_logits.float().cpu(), sp, BLANK,
            lengths=enc_lens.cpu().tolist(),
        )
        refs.extend(batch['texts'])
        hyps.extend(decoded)

        del teacher_h, teacher_mask, tokens, tok_lens, mel_lens
        del kl_logits, ctc_logits, loss_kl, loss_ctc, loss

    student.train()
    n = max(n_batches, 1)
    return {
        'loss':     total_loss / n,
        'loss_kl':  total_kl / n,
        'loss_ctc': total_ctc / n,
        'wer':      compute_wer(refs, hyps),
        'n_samples': len(refs),
    }


def train_stage(student, dataset, stage_name, epochs, lr, bs, ga,
                max_seconds, freeze_mamba=False, gc_norm=None, warmup=0,
                val_dataset=None):
    """
    Returns (best_loss, completed):
      completed=True: all `epochs` epochs finished in this call.
      completed=False: exited early due to session time budget. The full
                        state has been written to {stage_name}_latest.pt;
                        Rerun this cell in a new session to resume.

    Logs to the active wandb run with namespace `{stage_name}/...`:
      train/loss, train/loss_kl, train/loss_ctc, train/lr, train/grad_scale,
      train/gpu_mem_gb, train/elapsed_h   — every gradient update
      epoch_avg/loss, epoch_avg/loss_kl, epoch_avg/loss_ctc — end of epoch
      val/loss, val/loss_kl, val/loss_ctc, val/wer — end of epoch (if val_dataset)
    """

    # Route attribute access through .module if DataParallel-wrapped.
    _core = student.module if isinstance(student, torch.nn.DataParallel) else student
    if freeze_mamba:
        for p in _core.mamba.parameters():
            p.requires_grad = False
        trainable = sum(p.numel() for p in _core.parameters() if p.requires_grad)
        print(f'[{stage_name}] Mamba FROZEN — {trainable/1e6:.1f}M trainable')
    else:
        for p in _core.mamba.parameters():
            p.requires_grad = True
        trainable = sum(p.numel() for p in _core.parameters() if p.requires_grad)
        print(f'[{stage_name}] All params unfrozen — {trainable/1e6:.1f}M trainable')

    # CTC on a frozen backbone collapses to 'predict blank everywhere' — the
    # head can't learn alignment without gradients flowing into Mamba. Gate
    # the CTC term out during the frozen stage so it does pure feature
    # distillation (loss_kl), then re-enable it once Mamba unfreezes.
    eff_a_ctc = 0.0 if freeze_mamba else C.a_ctc
    # CTC warmup: ramp a_ctc from 0.05 to full value over first 500 steps
    # to prevent gradient explosion when transitioning from frozen→unfrozen.
    ctc_warmup_steps = 500 if not freeze_mamba else 0
    ctc_warmup_floor = 0.05
    print(f'[{stage_name}] loss = {C.a_kl} * loss_kl + {eff_a_ctc} * loss_ctc')
    if ctc_warmup_steps > 0:
        print(f'[{stage_name}] CTC warmup: {ctc_warmup_floor} → {eff_a_ctc} over {ctc_warmup_steps} steps')

    max_enc_frames = int(max_seconds * 100 / 2)  # 50 fps after teacher 2x downsample

    # We compute steps-per-epoch up front because LambdaLR needs total_steps
    # to schedule the cosine. With shard-bucketed batches, drop_last applies
    # PER SHARD, so the count is sum_over_shards(samples_per_shard // bs)
    # which is slightly less than len(dataset)//bs. Compute it exactly via
    # a throwaway sampler so the scheduler matches reality.
    _tmp_sampler = ShardBucketBatchSampler(
        dataset, batch_size=bs, generator=None, drop_last=True,
    )
    steps_per_epoch = len(_tmp_sampler) // ga
    del _tmp_sampler
    total_steps = max(1, epochs * steps_per_epoch)

    optimizer = torch.optim.AdamW(
        filter(lambda p: p.requires_grad, student.parameters()),
        lr=lr, weight_decay=0.01,
    )

    def lr_lambda(step):
        if step < warmup:
            return step / max(warmup, 1)
        progress = (step - warmup) / max(total_steps - warmup, 1)
        return 0.5 * (1 + math.cos(math.pi * progress))

    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
    scaler = torch.amp.GradScaler('cuda')

    # Resume state 
    ckpt_path = _save_path(stage_name)
    best_path = _save_path(stage_name, 'best')
    start_epoch = 0
    start_batch_idx = 0   # within-epoch resume offset
    global_step = 0
    best_loss = float('inf')

    # Early-stopping state. Configurable; safe defaults if Config doesn't
    # have these fields (back-compat with older Config classes).
    es_enabled    = getattr(C, 'es_enabled',   True) and (val_dataset is not None)
    es_patience   = getattr(C, 'es_patience',  3)
    es_min_delta  = getattr(C, 'es_min_delta', 1e-4)
    es_metric_key = getattr(C, 'es_metric',    'loss')  # 'loss' or 'wer'
    best_val_metric = float('inf')
    epochs_since_improvement = 0
    if val_dataset is None and getattr(C, 'es_enabled', True):
        print(f'[{stage_name}] Early stopping requested but no val_dataset '
              f'passed; disabled for this stage.')

    # Try writable dir first, then read-only baseline.
    ckpt, ckpt_src = try_load_full_checkpoint(_load_candidates(stage_name))
    if ckpt is not None:
        # Normalize: legacy checkpoints (single-GPU) have no `module.`
        # prefix; older DP-saved ones do. Strip if present, then load
        # into the un-wrapped model. Works whether or not we're using DP
        # this session.
        _unwrap(student).load_state_dict(_normalize_state_dict(ckpt['model']))
        optimizer.load_state_dict(ckpt['optimizer'])
        scheduler.load_state_dict(ckpt['scheduler'])
        scaler.load_state_dict(ckpt['scaler'])
        start_epoch = ckpt['epoch']
        start_batch_idx = ckpt.get('batch_idx', 0)
        global_step = ckpt['global_step']
        best_loss = ckpt.get('best_loss', float('inf'))
        # Restore early-stopping state across sessions.
        best_val_metric = ckpt.get('best_val_metric', float('inf'))
        epochs_since_improvement = ckpt.get('epochs_since_improvement', 0)
        if ckpt.get('early_stopped'):
            print(f'  (previously early-stopped — start_epoch={start_epoch} '
                  f'should equal epochs={epochs}, stage will exit fast)')
        is_baseline = (ckpt_src and ckpt_src.startswith(C.ckpt_load_dir))
        src_label = 'baseline (read-only)' if is_baseline else 'this-session save'
        print(f'Resumed {stage_name} from {src_label}: {ckpt_src}')
        print(f'  epoch {start_epoch+1}/{epochs}, batch_idx={start_batch_idx}, '
              f'step {global_step}, best_loss={best_loss:.4f}')
    elif stage_name == 'unfrozen':
        # Fresh stage 2: pull model weights from the completed stage 1
        # (also searches the read-only baseline).
        prev, prev_src = try_load_full_checkpoint(_load_candidates('frozen'))
        if prev is not None:
            _unwrap(student).load_state_dict(_normalize_state_dict(prev['model']))
            print(f'Loaded frozen-stage weights into unfrozen stage from {prev_src}')
            # The frozen stage's ctc_head was trained against a frozen backbone,
            # which lets CTC collapse into the 'predict blank everywhere' local
            # minimum. Carrying that collapsed head into stage 2 makes the loss
            # explode within a few hundred steps once Mamba unfreezes (saw
            # loss=inf @ step 2734 in the prior session). Reset to fresh init;
            # the train_proj + Mamba feature alignment from stage 1 is what we
            # actually wanted to inherit.
            _unwrap(student).ctc_head.reset_parameters()
            print('Reset ctc_head to fresh init (frozen-stage CTC was collapsed)')

    # If we already finished this stage in a previous session, exit fast.
    if start_epoch >= epochs:
        print(f'[{stage_name}] Already complete (epoch {start_epoch}/{epochs})')
        return best_loss, True

    student.train()
    last_save = time.monotonic()
    save_interval_s = CKPT_EVERY_MINUTES * 60

    for epoch in range(start_epoch, epochs):
        # Deterministic per-epoch shuffle — same epoch number always produces
        # the same sample order across resumes, so within-epoch skip works.
        epoch_gen = torch.Generator().manual_seed(C.seed * 1_000_003 + epoch)
        # Shard-bucketed sampling: same epoch_gen ensures deterministic
        # resume — same seed = same shard order = same batch sequence,
        # so within-epoch skip-N still lands on the same sample.
        train_sampler = ShardBucketBatchSampler(
            dataset, batch_size=bs, generator=epoch_gen, drop_last=True,
        )
        loader = DataLoader(
            dataset, batch_sampler=train_sampler,
            num_workers=C.workers, persistent_workers=(C.workers > 0),
            pin_memory=True,
            collate_fn=lambda b: collate_kd(b, max_enc_frames=max_enc_frames),
        )

        # Skip already-processed batches when resuming mid-epoch.
        skip_batches = start_batch_idx if epoch == start_epoch else 0
        if skip_batches:
            print(f'  Skipping first {skip_batches} batches of epoch {epoch+1} (resume)')
        # After this epoch, future epochs start from batch 0.
        start_batch_idx = 0

        epoch_loss = epoch_kl = epoch_ctc = 0.0
        n_batches = 0
        optimizer.zero_grad(set_to_none=True)

        pbar = tqdm(loader, desc=f'[{stage_name}] E{epoch+1}/{epochs}',
                    unit='batch', leave=True, initial=skip_batches,
                    total=len(loader))

        time_up = False

        for batch_idx, batch in enumerate(loader):
            if batch_idx < skip_batches:
                # Cheap skip: collate ran but we drop the result before any GPU work.
                pbar.update(0)
                continue
            pbar.update(1)

            teacher_h = batch['teacher_h'].to(device, non_blocking=True)
            teacher_mask = batch['teacher_mask'].to(device, non_blocking=True)
            tokens = batch['tokens'].to(device, non_blocking=True)
            tok_lens = batch['tok_lens'].to(device, non_blocking=True)
            mel_lens = batch['mel_lens'].to(device, non_blocking=True)

            # Use the true per-sample valid frame count from teacher_mask. The previous
            # `mel_lens // 2` assumed Whisper's exact 2x encoder stride and could
            # drift from the actual cached/padded length.
            enc_lens = teacher_mask.sum(dim=1).long().clamp(min=1)

            with torch.amp.autocast('cuda', dtype=torch.float16):
                kl_logits, ctc_logits = student(teacher_h, teacher_mask)
                loss_kl = kd_feature_loss(kl_logits, teacher_h, teacher_mask)
                loss_ctc = ctc_loss_fn(ctc_logits, tokens, enc_lens, tok_lens)
                # Apply CTC warmup ramp
                if ctc_warmup_steps > 0 and global_step < ctc_warmup_steps:
                    _ctc_w = ctc_warmup_floor + (eff_a_ctc - ctc_warmup_floor) * (global_step / ctc_warmup_steps)
                else:
                    _ctc_w = eff_a_ctc
                loss = (C.a_kl * loss_kl + _ctc_w * loss_ctc) / ga

            scaler.scale(loss).backward()

            if (batch_idx + 1) % ga == 0:
                if gc_norm:
                    scaler.unscale_(optimizer)
                    torch.nn.utils.clip_grad_norm_(student.parameters(), gc_norm)
                scale_before = scaler.get_scale()
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)
                if scaler.get_scale() >= scale_before:
                    scheduler.step()
                    global_step += 1

                    # Wandb log
                    _step_metrics = {
                        f'{stage_name}/train/loss': loss.item() * ga,
                        f'{stage_name}/train/loss_kl': loss_kl.item(),
                        f'{stage_name}/train/loss_ctc': loss_ctc.item(),
                        f'{stage_name}/train/lr': scheduler.get_last_lr()[0],
                        f'{stage_name}/train/grad_scale':  scaler.get_scale(),
                        f'{stage_name}/train/gpu_mem_gb':  torch.cuda.memory_allocated() / 1e9,
                        f'{stage_name}/train/elapsed_h': (time.monotonic() - SESSION_START) / 3600,
                        f'{stage_name}/train/global_step': global_step,
                        f'{stage_name}/train/epoch_frac': epoch + (batch_idx + 1) / max(len(loader), 1),
                    }
                    wandb_run.log(_step_metrics)
                    # Mirror to JSONL on disk (full fidelity).
                    loss_log.log(
                        'train_step',
                        stage=stage_name,
                        epoch=epoch,
                        batch_idx=batch_idx,
                        global_step=global_step,
                        loss=loss.item() * ga,
                        loss_kl=loss_kl.item(),
                        loss_ctc=loss_ctc.item(),
                        lr=scheduler.get_last_lr()[0],
                        grad_scale=scaler.get_scale(),
                        gpu_mem_gb=torch.cuda.memory_allocated() / 1e9,
                    )
                else:
                    loss_log.log(
                        'scaler_skip',
                        stage=stage_name,
                        epoch=epoch,
                        batch_idx=batch_idx,
                        scale_before=scale_before,
                        scale_after=scaler.get_scale(),
                    )
                    if batch_idx % C.log_every == 0:
                        pbar.write(f'  [scaler skip] inf/nan grads; scale '
                                   f'{scale_before:g} -> {scaler.get_scale():g}')

            epoch_loss += loss.item() * ga
            epoch_kl += loss_kl.item()
            epoch_ctc += loss_ctc.item()
            n_batches += 1

            del teacher_h, teacher_mask, tokens, tok_lens, mel_lens
            del kl_logits, ctc_logits, loss_kl, loss_ctc, loss

            now = time.monotonic()

            # Rolling save every CKPT_EVERY_MINUTES of wall time.
            if now - last_save >= save_interval_s:
                save_full_checkpoint(
                    ckpt_path, model=student, optimizer=optimizer,
                    scheduler=scheduler, scaler=scaler,
                    stage_name=stage_name, epoch=epoch,
                    batch_idx=batch_idx + 1,        # next batch to run
                    global_step=global_step, best_loss=best_loss,
                    best_val_metric=best_val_metric,
                    epochs_since_improvement=epochs_since_improvement)
                last_save = now
                pbar.write(f'saved {os.path.basename("checkpoints/")} '
                           f'@ step {global_step} (time left: {fmt_hms(time_left_seconds())})')

            # Time budget exit: save and bail before Kaggle kills us.
            if now >= SESSION_DEADLINE:
                save_full_checkpoint(
                    ckpt_path, model=student, optimizer=optimizer,
                    scheduler=scheduler, scaler=scaler,
                    stage_name=stage_name, epoch=epoch,
                    batch_idx=batch_idx + 1,
                    global_step=global_step, best_loss=best_loss,
                    best_val_metric=best_val_metric,
                    epochs_since_improvement=epochs_since_improvement)
                pbar.write(f'\nSession budget reached. Saved checkpoint to '
                           f'{ckpt_path}.\n   Click "Save Version" to commit '
                           f'/kaggle/working, then resume in a new session.')
                time_up = True
                break

            if batch_idx % C.log_every == 0:
                pbar.set_postfix_str(
                    f'loss={epoch_loss/max(n_batches,1):.3f} '
                    f'mem={torch.cuda.memory_allocated()/1e9:.1f}G '
                    f'lr={scheduler.get_last_lr()[0]:.2e} '
                    f'left={fmt_hms(time_left_seconds())}')

        pbar.close()

        if time_up:
            return best_loss, False

        # End-of-epoch bookkeeping
        avg_loss = epoch_loss / max(n_batches, 1)
        avg_kl = epoch_kl / max(n_batches, 1)
        avg_ctc = epoch_ctc / max(n_batches, 1)
        print(f'  E{epoch+1} avg — loss={avg_loss:.4f} kl={avg_kl:.4f} ctc={avg_ctc:.4f}')

        # Build the end-of-epoch log payload. We combine epoch_avg + val into
        # one wandb.log() call so they sit at the same wandb step.
        epoch_payload = {
            f'{stage_name}/epoch_avg/loss': avg_loss,
            f'{stage_name}/epoch_avg/loss_kl': avg_kl,
            f'{stage_name}/epoch_avg/loss_ctc': avg_ctc,
            f'{stage_name}/epoch_avg/epoch': epoch + 1,
            f'{stage_name}/epoch_avg/global_step': global_step,
        }

        # Validation pass — only if val_dataset provided and time permits.
        # Cap val batches so eval stays under ~2 min even on slow_forward.
        if val_dataset is not None and time_left_seconds() > 300:
            val_max_batches = min(len(val_dataset) // 2, 250)
            print(f'  Running validation on up to {val_max_batches * 2} samples...')
            val_metrics = evaluate(
                student, val_dataset,
                max_seconds=max_seconds,
                bs=2,
                max_batches=val_max_batches,
                a_ctc=eff_a_ctc,
            )
            print(f'  val — loss={val_metrics["loss"]:.4f}  '
                  f'wer={val_metrics["wer"]*100:.2f}%  '
                  f'(n={val_metrics["n_samples"]})')
            epoch_payload.update({
                f'{stage_name}/val/loss':     val_metrics['loss'],
                f'{stage_name}/val/loss_kl':  val_metrics['loss_kl'],
                f'{stage_name}/val/loss_ctc': val_metrics['loss_ctc'],
                f'{stage_name}/val/wer':      val_metrics['wer'],
                f'{stage_name}/val/epoch':    epoch + 1,
            })
            torch.cuda.empty_cache()

            # ── Early stopping ───────────────────────────────────────
            if es_enabled:
                current_val = (val_metrics['wer'] if es_metric_key == 'wer'
                               else val_metrics['loss'])
                improved = current_val < best_val_metric - es_min_delta
                if improved:
                    epochs_since_improvement = 0
                    best_val_metric = current_val
                    print(f'  early-stop: val/{es_metric_key}={current_val:.5f} '
                          f'(best so far) — counter reset')
                else:
                    epochs_since_improvement += 1
                    print(f'  early-stop: val/{es_metric_key}={current_val:.5f} '
                          f'(best={best_val_metric:.5f}) — '
                          f'no improvement {epochs_since_improvement}/{es_patience}')
                epoch_payload[f'{stage_name}/val/best_{es_metric_key}'] = best_val_metric
                epoch_payload[f'{stage_name}/val/epochs_without_improvement'] = (
                    epochs_since_improvement
                )

        wandb_run.log(epoch_payload)
        # Mirror to JSONL.
        loss_log.log(
            'epoch_end',
            stage=stage_name,
            epoch=epoch + 1,
            global_step=global_step,
            train_loss=avg_loss,
            train_loss_kl=avg_kl,
            train_loss_ctc=avg_ctc,
            val_loss=epoch_payload.get(f'{stage_name}/val/loss'),
            val_loss_kl=epoch_payload.get(f'{stage_name}/val/loss_kl'),
            val_loss_ctc=epoch_payload.get(f'{stage_name}/val/loss_ctc'),
            val_wer=epoch_payload.get(f'{stage_name}/val/wer'),
        )

        # Mark this epoch as fully done by saving with epoch+1 (so resume
        # starts at the NEXT epoch from batch 0).
        save_full_checkpoint(
            ckpt_path, model=student, optimizer=optimizer,
            scheduler=scheduler, scaler=scaler,
            stage_name=stage_name, epoch=epoch + 1,
            batch_idx=0, global_step=global_step,
            best_loss=min(best_loss, avg_loss),
            best_val_metric=best_val_metric,
            epochs_since_improvement=epochs_since_improvement)
        last_save = time.monotonic()

        if avg_loss < best_loss:
            best_loss = avg_loss
            save_best_model_only(best_path, student, epoch=epoch,
                                 global_step=global_step, loss=best_loss)
            print(f'  ★ New best: {best_loss:.4f}')

        # ── Early-stopping exit ─────────────────────────────────────
        # Triggered once we've gone `patience` epochs without improvement
        # on the chosen val metric. We mark the stage complete (epoch=epochs)
        # so that any future session sees `start_epoch >= epochs` and
        # skips this stage entirely.
        if es_enabled and epochs_since_improvement >= es_patience:
            print(f'\n[{stage_name}] EARLY STOPPING at epoch {epoch+1}/{epochs} '
                  f'— no improvement in val/{es_metric_key} for '
                  f'{epochs_since_improvement} epochs (best={best_val_metric:.5f}).')
            save_full_checkpoint(
                ckpt_path, model=student, optimizer=optimizer,
                scheduler=scheduler, scaler=scaler,
                stage_name=stage_name, epoch=epochs,  # mark as complete
                batch_idx=0, global_step=global_step,
                best_loss=best_loss,
                best_val_metric=best_val_metric,
                epochs_since_improvement=epochs_since_improvement,
                early_stopped=True,
            )
            loss_log.log(
                'early_stop',
                stage=stage_name,
                epoch=epoch + 1,
                best_loss=best_loss,
                best_val_metric=best_val_metric,
            )
            return best_loss, True

    print(f'[{stage_name}] Done — best_loss={best_loss:.4f}\n')
    return best_loss, True


# 9. Run stages 
print('\n' + '='*60)
print(f'STAGE 1: Frozen Mamba backbone  (time left: {fmt_hms(time_left_seconds())})')
print('='*60)
s1_loss, s1_done = train_stage(
    student, train_ds,
    stage_name='frozen',
    epochs=C.fr_epochs,
    lr=C.fr_lr,
    bs=C.fr_bs,
    ga=C.fr_ga,
    max_seconds=C.max_s1,
    freeze_mamba=True,
    val_dataset=val_ds,
)
flush()

if not s1_done:
    print('\n⏸  Stage 1 paused. Save Version  : reopen  : run again to resume.')
    loss_log.log('session_pause', stage='frozen', completed=False)
    loss_log.close()
    wandb_run.finish(exit_code=0, quiet=True)
elif time_left_seconds() < 600:
    print('\n⏸  Stage 1 done but < 10 min left. Save Version and continue '
          'stage 2 next session.')
    loss_log.log('session_pause', stage='frozen', completed=True)
    loss_log.close()
    wandb_run.finish(exit_code=0, quiet=True)
else:
    print('\n' + '='*60)
    print(f'STAGE 2: Full fine-tuning  (time left: {fmt_hms(time_left_seconds())})')
    print('='*60)
    s2_loss, s2_done = train_stage(
        student, train_ds,
        stage_name='unfrozen',
        epochs=C.un_epochs,
        lr=C.un_lr,
        bs=C.un_bs,
        ga=C.un_ga,
        max_seconds=C.max_s2,
        freeze_mamba=False,
        gc_norm=C.gc_norm,
        warmup=C.warmup,
        val_dataset=val_ds,
    )
    if s2_done:
        print(f'\nTraining complete — Stage1={s1_loss:.4f}  Stage2={s2_loss:.4f}')
        wandb_run.summary['final/stage1_loss'] = s1_loss
        wandb_run.summary['final/stage2_loss'] = s2_loss
        loss_log.log('training_complete', stage1_loss=s1_loss, stage2_loss=s2_loss)
        loss_log.close()
        wandb_run.finish(exit_code=0, quiet=True)
    else:
        print(f'\n⏸  Stage 2 paused at loss {s2_loss:.4f}. Save Version  : resume.')
        loss_log.log('session_pause', stage='unfrozen', completed=False, last_loss=s2_loss)
        loss_log.close()
        wandb_run.finish(exit_code=0, quiet=True)

flush()

### Test-set evaluation (run after training completes)

Loads the best checkpoint and runs a two-pass evaluation on the held-out test set saved to `splits.json`:

**Pass 1 — accuracy** (full test set, bs=2): WER + CER via `jiwer`. Saves per-sample `{ref, hyp, audio_seconds}` to `test_results/predictions_<stage>.jsonl`.

**Pass 2 — latency** (bs=1, with warmup + `torch.cuda.synchronize`): mean, p50, p95, p99 in ms. Real-time factor (RTF) if `audio_seconds` is in the batch.

To run other methods on the same test set, share these three files: `splits.json`, `test_results/predictions_<stage>.jsonl`, `test_results/results_<stage>.json`. Same indices → same audio → directly comparable WER/CER/latency.

This cell is **independent of training state** — it rebuilds the student from disk and can be run any time after at least one stage has saved a `*_best.pt` checkpoint.


In [ ]:
# ── Test-set evaluation: WER + CER + Latency ──────────────────────
# Independent of the training cell's in-memory state — rebuilds the
# student from disk so it works even after a kernel restart.
import json, time
import numpy as np
import jiwer
from torch.utils.data import Subset, DataLoader

# 1. Load splits (test indices).
SPLITS_PATH = os.path.join(C.ckpt_dir, 'splits.json')
assert os.path.exists(SPLITS_PATH), (
    f'No splits.json at {SPLITS_PATH}. Run the training cell at least '
    'once first — it creates the splits.'
)
with open(SPLITS_PATH) as f:
    splits = json.load(f)

# 2. Ensure dataset is loaded (might not be if kernel was restarted).
if 'ds' not in globals():
    print('Re-loading dataset...')
    ds = KDDataset(C.cache_dir, sp)
test_ds = Subset(ds, splits['test_indices'])
print(f'Test set: {len(test_ds):,} samples '
      f'(fingerprint={splits.get("fingerprint")})')

# 3. Find best checkpoint — prefer unfrozen, fall back to frozen.
candidates = [
    ('unfrozen', os.path.join(C.ckpt_dir, 'unfrozen_best.pt')),
    ('unfrozen', os.path.join(C.ckpt_dir, 'unfrozen_latest.pt')),
    ('frozen', os.path.join(C.ckpt_dir, 'frozen_best.pt')),
    ('frozen', os.path.join(C.ckpt_dir, 'frozen_latest.pt')),
]
stage, best_path = None, None
for s, p in candidates:
    if os.path.exists(p):
        stage, best_path = s, p
        break
assert best_path is not None, (
    f'No checkpoint found under {C.ckpt_dir}. Train at least one stage first.'
)
print(f'Using checkpoint: {best_path} (stage={stage})')

# 4. Build a fresh student and load weights.
# Always load into the un-wrapped model; strip DataParallel `module.` prefix
# from saved state-dict if present.
test_student = StudentASRv2(C).to(device)
_state = torch.load(best_path, map_location='cpu', weights_only=False)
_sd = _state['model']
# Normalize the on-disk state-dict to no-prefix form. New checkpoints
# (post-v7) are already in this form, so this is a no-op then. Legacy
# DP-saved checkpoints have `module.` prefixes which we strip. The old
# inline comprehension had a bug — it filtered out non-prefixed keys,
# which would silently zero-init half the model on mixed dicts.
if any(k.startswith('module.') for k in _sd):
    _sd = {(k[len('module.'):] if k.startswith('module.') else k): v
           for k, v in _sd.items()}
test_student.load_state_dict(_sd)
test_student.eval()
print(f'  checkpoint meta: epoch={_state.get("epoch")}, '
      f'global_step={_state.get("global_step")}, '
      f'loss={_state.get("loss", _state.get("best_loss", float("nan"))):.4f}')

# 5. Set up output paths.
results_dir = os.path.join(C.ckpt_dir, 'test_results')
os.makedirs(results_dir, exist_ok=True)
pred_path = os.path.join(results_dir, f'predictions_{stage}.jsonl')
results_path = os.path.join(results_dir, f'results_{stage}.json')

# ── Pass 1: accuracy (WER + CER over full test set, bs=2) ───────
print(f'\nPass 1/2: WER + CER on {len(test_ds):,} test samples...')
max_seconds = C.max_s2
max_enc_frames = int(max_seconds * 100 / 2)

acc_sampler = ShardBucketBatchSampler(
    test_ds, batch_size=2, drop_last=False, shuffle=False,
)
acc_loader = DataLoader(
    test_ds, batch_sampler=acc_sampler,
    num_workers=C.workers, persistent_workers=False,
    pin_memory=True,
    collate_fn=lambda b: collate_kd(b, max_enc_frames=max_enc_frames),
)

refs, hyps, audio_seconds_list = [], [], []
with open(pred_path, 'w') as pf, torch.no_grad():
    for batch in tqdm(acc_loader, desc='accuracy', unit='batch'):
        teacher_h = batch['teacher_h'].to(device, non_blocking=True)
        teacher_mask = batch['teacher_mask'].to(device, non_blocking=True)
        enc_lens = teacher_mask.sum(dim=1).long().clamp(min=1)

        with torch.amp.autocast('cuda', dtype=torch.float16):
            _kl, ctc_logits = test_student(teacher_h, teacher_mask)

        decoded = greedy_ctc_decode(
            ctc_logits.float().cpu(), sp, BLANK,
            lengths=enc_lens.cpu().tolist(),
        )
        batch_secs = batch.get('audio_seconds', [None] * len(decoded))
        for ref, hyp, sec in zip(batch['texts'], decoded, batch_secs):
            refs.append(ref)
            hyps.append(hyp)
            sec_val = float(sec) if sec is not None else None
            if sec_val is not None:
                audio_seconds_list.append(sec_val)
            pf.write(json.dumps(
                {'ref': ref, 'hyp': hyp, 'audio_seconds': sec_val},
                ensure_ascii=False,
            ) + '\n')

# Compute WER and CER. Same jiwer text-normalization is applied to both.
wer = jiwer.wer(refs, hyps)
cer = jiwer.cer(refs, hyps)
print(f'  WER = {wer*100:.2f}%   CER = {cer*100:.2f}%   (n={len(refs)})')

# ── Pass 2: latency at bs=1 over a subset, with proper warmup ────
N_LAT = min(200, len(test_ds))
print(f'\nPass 2/2: latency over {N_LAT} samples (bs=1, fp16)...')

lat_subset = Subset(test_ds, list(range(N_LAT)))
lat_loader = DataLoader(
    lat_subset, batch_size=1, shuffle=False, num_workers=2,
    pin_memory=True, drop_last=False,
    collate_fn=lambda b: collate_kd(b, max_enc_frames=max_enc_frames),
)

# Warmup (first few forwards trigger Triton JIT / kernel caching).
with torch.no_grad():
    for i, batch in enumerate(lat_loader):
        if i >= 10:
            break
        th = batch['teacher_h'].to(device, non_blocking=True)
        tm = batch['teacher_mask'].to(device, non_blocking=True)
        with torch.amp.autocast('cuda', dtype=torch.float16):
            test_student(th, tm)
        torch.cuda.synchronize()

# Measure — torch.cuda.synchronize() before and after to exclude
# data-load + queue time from the measurement.
latencies_ms, sample_audio_secs = [], []
with torch.no_grad():
    for batch in tqdm(lat_loader, desc='latency', unit='sample'):
        th = batch['teacher_h'].to(device, non_blocking=True)
        tm = batch['teacher_mask'].to(device, non_blocking=True)
        sec = batch.get('audio_seconds', [None])
        sec_val = float(sec[0]) if sec and sec[0] is not None else None

        torch.cuda.synchronize()
        t0 = time.perf_counter()
        with torch.amp.autocast('cuda', dtype=torch.float16):
            test_student(th, tm)
        torch.cuda.synchronize()
        latencies_ms.append((time.perf_counter() - t0) * 1000)
        sample_audio_secs.append(sec_val)

lat_arr = np.array(latencies_ms)

# Real-time factor: latency_seconds / audio_seconds. <1.0 is faster than
# real-time. Reported only if audio_seconds is available.
rtf_stats = None
if all(s is not None for s in sample_audio_secs) and sample_audio_secs:
    sec_arr = np.array(sample_audio_secs)
    rtf = (lat_arr / 1000.0) / sec_arr
    rtf_stats = {
        'mean': float(rtf.mean()),
        'p50':  float(np.percentile(rtf, 50)),
        'p95':  float(np.percentile(rtf, 95)),
    }

# Assemble result blob.
import transformers as _transformers_mod
results = {
    'stage':           stage,
    'checkpoint_path': best_path,
    'checkpoint_meta': {
        'epoch':       _state.get('epoch'),
        'global_step': _state.get('global_step'),
        'loss':        float(_state.get('loss', _state.get('best_loss', float('nan')))),
    },
    'n_samples_accuracy': len(refs),
    'n_samples_latency':  len(latencies_ms),
    'wer': float(wer),
    'cer': float(cer),
    'latency_ms': {
        'mean': float(lat_arr.mean()),
        'std':  float(lat_arr.std()),
        'p50':  float(np.percentile(lat_arr, 50)),
        'p95':  float(np.percentile(lat_arr, 95)),
        'p99':  float(np.percentile(lat_arr, 99)),
        'min':  float(lat_arr.min()),
        'max':  float(lat_arr.max()),
    },
    'system': {
        'gpu_name':              torch.cuda.get_device_name(0),
        'precision':             'fp16 (autocast)',
        'torch_version':         torch.__version__,
        'transformers_version':  _transformers_mod.__version__,
    },
    'config': {
        'batch_size_accuracy':  2,
        'batch_size_latency':   1,
        'max_audio_seconds':    max_seconds,
        'student_params_M':     sum(p.numel() for p in test_student.parameters()) / 1e6,
        'note':                 'Latency = student forward only; teacher_h precomputed.',
    },
    'splits': {
        'splits_path':  SPLITS_PATH,
        'fingerprint':  splits.get('fingerprint'),
        'seed':         splits.get('seed'),
        'test_size':    splits.get('test_size'),
    },
}
if rtf_stats is not None:
    results['rtf'] = rtf_stats

with open(results_path, 'w') as f:
    json.dump(results, f, indent=2)

# ── Print summary ──
print(f'\n{"=" * 60}')
print(f'TEST SET EVALUATION ({stage})')
print(f'{"=" * 60}')
print(f'  WER          : {results["wer"]*100:7.2f}%')
print(f'  CER          : {results["cer"]*100:7.2f}%')
print(f'  Latency mean : {results["latency_ms"]["mean"]:7.1f} ms')
print(f'  Latency p50  : {results["latency_ms"]["p50"]:7.1f} ms')
print(f'  Latency p95  : {results["latency_ms"]["p95"]:7.1f} ms')
print(f'  Latency p99  : {results["latency_ms"]["p99"]:7.1f} ms')
if rtf_stats is not None:
    print(f'  RTF mean     : {rtf_stats["mean"]:7.3f}  '
          '(student forward only; <1.0 = faster than real-time)')
print(f'\nArtifacts saved (use these to compare other methods on the same set):')
print(f'  splits.json      : {SPLITS_PATH}')
print(f'  predictions.jsonl: {pred_path}')
print(f'  results.json     : {results_path}')

del test_student
torch.cuda.empty_cache()


### CTC output diagnostic (run if WER stays at 100%)

After training has saved at least one checkpoint, this cell loads it, runs the model on 10 validation samples, and prints the per-frame argmax distribution. WER=100% has four possible causes — this output tells you which:

- **Blank-dominant collapse** (>95% of frames argmax to BLANK): expected during frozen stage. The CTC head can't overcome the trivial "predict blank everywhere" local minimum without backbone gradients. Not a bug — move to unfrozen stage.
- **Mode collapse** on a single non-blank token: training instability, often a vocab/temperature issue.
- **Diverse outputs but wrong text**: model is learning, just hasn't converged yet on alignment.
- **Tokenizer mismatch**: token IDs look right but decoded text doesn't match — bug in the encode/decode round-trip.


In [ ]:
# ── CTC output diagnostic ─────────────────────────────────────
# Distinguishes blank-collapse vs mode-collapse vs genuine slow learning vs
# tokenizer bug. Run after at least one stage has saved a checkpoint.
from torch.utils.data import Subset, DataLoader
from collections import Counter

ckpt_path = os.path.join(C.ckpt_dir, 'frozen_latest.pt')
if not os.path.exists(ckpt_path):
    raise FileNotFoundError(f'No checkpoint at {ckpt_path}. Train at least '
                            'partway through the frozen stage first.')

# Build a fresh student so we don't disturb the in-memory training state.
diag = StudentASRv2(C).to(device)
_state = torch.load(ckpt_path, map_location='cpu', weights_only=False)
_sd = _state['model']
if any(k.startswith('module.') for k in _sd):
    _sd = {(k[len('module.'):] if k.startswith('module.') else k): v
           for k, v in _sd.items()}
diag.load_state_dict(_sd)
diag.eval()
print(f'Loaded {ckpt_path}')
print(f'  epoch={_state.get("epoch")}, step={_state.get("global_step")}, '
      f'best_loss={_state.get("best_loss", float("nan")):.4f}')

# Reference summary of vocab structure.
print(f'\nVocab structure:')
print(f'  sp.GetPieceSize()  = {sp.GetPieceSize()}  (SPM token IDs 0..{sp.GetPieceSize()-1})')
print(f'  BLANK              = {BLANK}  (CTC blank index)')
print(f'  C.vocab_size       = {C.vocab_size}  (ctc_head output dim)')
assert BLANK == sp.GetPieceSize(), "BLANK should be one past the SPM range"
assert C.vocab_size == BLANK + 1, "ctc_head must output (vocab + 1) classes"

# Pull 10 val samples from the persisted splits.
SPLITS_PATH = os.path.join(C.ckpt_dir, 'splits.json')
with open(SPLITS_PATH) as f:
    splits_data = json.load(f)
val_subset = Subset(ds, splits_data['val_indices'][:10])
loader = DataLoader(
    val_subset, batch_size=1, shuffle=False, num_workers=0,
    collate_fn=lambda b: collate_kd(b, max_enc_frames=int(C.max_s1 * 100 / 2)),
)

print('\n' + '─' * 80)

all_argmax = Counter()
for i, batch in enumerate(loader):
    th = batch['teacher_h'].to(device)
    tm = batch['teacher_mask'].to(device)
    enc_lens = tm.sum(dim=1).long().clamp(min=1)
    valid_len = enc_lens[0].item()
    n_tokens = batch['tok_lens'][0].item()

    with torch.no_grad(), torch.amp.autocast('cuda', dtype=torch.float16):
        _kl, ctc_logits = diag(th, tm)

    # Per-frame argmax (after masking to valid frames).
    argmax_ids = ctc_logits[0, :valid_len].float().argmax(dim=-1).cpu().tolist()
    counter = Counter(argmax_ids)
    all_argmax.update(counter)

    # First-frame top-5 — shows whether non-blank tokens are even in contention.
    probs = ctc_logits[0, 0].float().softmax(dim=-1)
    top5 = probs.topk(5)

    ref = batch['texts'][0]
    decoded = greedy_ctc_decode(
        ctc_logits.float().cpu(), sp, BLANK, lengths=[valid_len],
    )[0]
    blank_pct = 100 * counter.get(BLANK, 0) / valid_len

    print(f'\n[{i}] frames={valid_len}  target_tokens={n_tokens}  '
          f'blank_pct={blank_pct:5.1f}%')
    print(f'    REF: {ref[:100]}')
    print(f'    HYP: {decoded[:100] if decoded else "(empty — only blanks)"}')
    print(f'    argmax top-3: ', end='')
    for tid, cnt in counter.most_common(3):
        piece = sp.id_to_piece(tid) if tid < sp.GetPieceSize() else f'<BLANK>'
        print(f'{piece}({cnt})  ', end='')
    print()
    print(f'    frame-0 top-5: ', end='')
    for j in range(5):
        tid = top5.indices[j].item()
        prob = top5.values[j].item()
        piece = sp.id_to_piece(tid) if tid < sp.GetPieceSize() else '<BLANK>'
        print(f'{piece}({prob:.2f})  ', end='')
    print()

print('\n' + '═' * 80)
print('AGGREGATE across 10 samples:')
total = sum(all_argmax.values())
print(f'  Total frames examined: {total}')
print(f'  Top-10 most-emitted tokens:')
top10 = all_argmax.most_common(10)
for tid, count in top10:
    piece = sp.id_to_piece(tid) if tid < sp.GetPieceSize() else '<BLANK>'
    marker = '  <-- BLANK' if tid == BLANK else ''
    print(f'    id={tid:5d}  {piece:25s} {count:7d}  ({100*count/total:5.1f}%){marker}')

# Verdict
print('\n' + '═' * 80)
print('DIAGNOSIS:')
top1_id, top1_count = top10[0]
top1_pct = 100 * top1_count / total
blank_pct = 100 * all_argmax.get(BLANK, 0) / total

if blank_pct > 95:
    print(f'  (a) BLANK-DOMINANT COLLAPSE  ({blank_pct:.1f}% blanks)')
    print()
    print('  This is expected when CTC trains on a FROZEN backbone — the')
    print('  head can\'t learn alignment without Mamba gradients. The CTC')
    print('  loss falls into the "predict blank everywhere" local minimum')
    print('  because blank IS statistically the right answer for most of')
    print('  the ~250 encoder frames vs ~10 target tokens.')
    print()
    print('  RECOMMENDED FIX: skip the rest of the frozen stage and move')
    print('  to unfrozen. The frozen stage was useful for feature alignment')
    print('  (loss_kl is already ~0.004), but cannot teach CTC alignment.')
elif top1_pct > 50 and top1_id != BLANK:
    piece = sp.id_to_piece(top1_id)
    print(f'  (b) MODE COLLAPSE on token "{piece}" (id={top1_id}, {top1_pct:.1f}%)')
    print()
    print('  Model emits the same non-blank token at most positions.')
    print('  Likely causes: vocab off-by-one, CTC target alignment bug,')
    print('  or a single training batch with extreme gradient dominated init.')
elif blank_pct < 80:
    print(f'  (c) DIVERSE OUTPUT  (blank only {blank_pct:.1f}%, top-1 non-blank '
          f'is "{sp.id_to_piece(top1_id) if top1_id != BLANK else "<BLANK>"}")')
    print()
    print('  Model is emitting varied tokens — alignment is forming.')
    print('  WER=100% means the decoded text doesn\'t match the reference yet.')
    print('  Keep training; this should improve in the unfrozen stage.')
else:
    print(f'  (d) MIXED PATTERN  (blanks {blank_pct:.1f}%, top non-blank '
          f'{top1_pct:.1f}%) — inspect per-sample HYP outputs above.')

# Bonus diagnostic: feature distillation health.
# loss_kl ~0.004 is suspicious — verify the student\'s kl_logits is actually
# varying across frames rather than predicting the per-sample mean.
print('\n' + '─' * 80)
print('Bonus: kl_logits diversity check (verify it\'s not just predicting mean):')
# diag.forward keeps teacher_h in fp16 and relies on autocast to bridge to the
# fp32 train_proj weights (see the comment on line 631 of cell 24). The main
# loop above uses autocast for the same reason; this block must too, otherwise
# train_proj raises "mat1 and mat2 must have the same dtype, but got Half and Float".
with torch.no_grad(), torch.amp.autocast('cuda', dtype=torch.float16):
    batch = next(iter(loader))
    th = batch['teacher_h'].to(device)
    tm = batch['teacher_mask'].to(device)
    kl_logits, _ = diag(th, tm)
    valid_len = tm.sum(dim=1)[0].item()
    student_var = kl_logits[0, :valid_len].float().var(dim=0).mean().item()
    teacher_var = th[0, :valid_len].float().var(dim=0).mean().item()
    print(f'  per-frame variance (avg over feature dim):')
    print(f'    teacher_h    : {teacher_var:.8e}')
    print(f'    student_kl   : {student_var:.8e}  '
          f'(ratio {student_var/max(teacher_var,1e-9):.2f})')
    if student_var < 0.1 * teacher_var:
        print('  WARN: student_kl variance << teacher. The low loss_kl may be')
        print('        partially from predicting the per-sample mean rather than')
        print('        frame-level details. Not catastrophic but worth noting.')

del diag
torch.cuda.empty_cache()
